<a href="https://colab.research.google.com/github/crangarita/sistema-recomendador/blob/master/Sistema_Recomendador_Multi_LLM_Claudia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sistema de Recomendación Multi-LLM con FCD (Arquitectura Completa)
**Autores:** Nelson Vivas, Julio Vasquez
**Fecha:** Noviembre 2025

Este Jupyter Notebook implementa paso a paso la arquitectura completa del sistema de recomendación híbrido. A diferencia de implementaciones simples, este sistema utiliza:

1.  **Arquitectura Agnóstica al Modelo:** Capa de abstracción para intercambiar LLMs (GPT-4o, Claude 3.5, Gemini 2.0) sin cambiar código.
2.  **RAG Optimizado (Nivel Producción):** Base de datos vectorial (ChromaDB) con parámetros HNSW ajustados (`search_ef=100`) y caché LRU para embeddings.
3.  **Ranking Avanzado:** Fusión de Rankings Recíprocos (RRF) y arquitectura híbrida de 2 etapas (Relevancia $\to$ Personalización).
4.  **Validación Semántica Triple:** Control de calidad estricto para prevenir alucinaciones.

## 1. Configuración e Importaciones
Instalamos las dependencias críticas, incluyendo `rapidfuzz` para matching de texto optimizado y `chromadb` para el almacenamiento vectorial.

In [ ]:
!pip install -q chromadb sentence-transformers pandas numpy openai anthropic google-generativeai rapidfuzz python-dotenv scipy scikit-learn requests

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.27.1 requires opentelemetry-exporter-otlp-proto-http>=1.36.0, which is not installed.
google-adk 1.27.1 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.40.0 which is incompatible.
google-adk 1.27.1 requires opentelemetry-sdk<1.39.0,>=1.36.0, but you have opentelemetry-sdk 1.40.0 which is incompatible.
opentelemetry-exporter-gcp-logging 1.11.0a0 requires opentelemetry-sdk<1.39.0,>=1.35.0, but you have opentelemetry-sdk 1.40.0 which is incompatible.


In [ ]:
!pip uninstall -y opentelemetry-exporter-otlp-proto-grpc opentelemetry-exporter-otlp-proto-http
!pip install -U google-genai -q

Found existing installation: opentelemetry-exporter-otlp-proto-grpc 1.40.0
Uninstalling opentelemetry-exporter-otlp-proto-grpc-1.40.0:
  Successfully uninstalled opentelemetry-exporter-otlp-proto-grpc-1.40.0


In [ ]:
import os
import json
import time
import hashlib
import numpy as np
import pandas as pd
import re
import random
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
from sentence_transformers import SentenceTransformer
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity
import chromadb
# Importar SDKs de proveedores
import openai
import anthropic
#import google.generativeai as genai
from google import genai
from google.genai import types
import shutil
import chromadb
from chromadb.config import Settings
import hashlib

In [ ]:
MAX_TOKENS = 1500  # Aumentado para JSON con descriptions
TEMPERATURE = 0.7
RETRY_ATTEMPTS = 3
TOP_K_CANDIDATES = 200  # Candidatos enviados a LLMs (expandido para mas contexto)

# Modo verbose: True = logs detallados, False = solo resultados importantes
VERBOSE = False  # Cambiar a True para ver logs completos

# ===============================================
# OPTIMIZACIONES ADICIONALES (Configurables)
# ===============================================
USE_PARALLEL_LLMS = True  # True = LLMs en paralelo (ThreadPoolExecutor), False = secuencial
USE_RAPIDFUZZ = True      # True = RapidFuzz (10x mas rapido), False = difflib
USE_VECTORIZED_MMR = True # True = MMR vectorizado (5x mas rapido), False = loop manual

print("Cargando encoder...")
encoder = SentenceTransformer('all-mpnet-base-v2')  # 768 dims
print("Encoder cargado (768 dimensiones)")
_query_embedding_cache = {}
_CACHE_MAX_SIZE = 1000

Cargando encoder...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoder cargado (768 dimensiones)


## ⚙️ Configuración de Entorno y Persistencia de Datos

Esta celda inicializa la infraestructura del proyecto, adaptándose automáticamente al entorno de ejecución (**Google Colab** vs. **Local**). Su objetivo principal es asegurar que tanto los datasets crudos como las bases de datos vectoriales (ChromaDB) estén disponibles antes de iniciar.

**Funcionalidades principales:**

1.  **Detección de Entorno:** Determina si el notebook corre en la nube o localmente para ajustar las rutas de acceso (`PATHS`).
2.  **Carga Optimizada de ChromaDB:**
    * *En Colab:* Monta Google Drive y busca archivos comprimidos (`.tar.gz`) que contienen los índices vectoriales ya calculados. Los extrae al entorno local de la sesión para evitar el costoso proceso de re-embedding (indexación).
    * *En Local:* Utiliza los directorios persistentes del entorno de desarrollo.
3.  **Gestión de Datasets (ETL):** Verifica la existencia de los archivos fuente (`movies.csv`, `spotify.csv`). Si no se encuentran, intenta descargarlos o copiarlos desde la carpeta compartida en Drive.

In [ ]:
DATASETS = {
    'serious_games': {
        'file': None,
        'columns': {
            'id': 'item_id',
            'title': 'title',
            'description': 'description'
        },
        'embedding_fields': ['title', 'description', 'item_type', 'type_label'],
        'top_k': 12,
        'threshold': 0.55,
        'external_threshold': 0.50,
        'mmr_lambda': 0.75,
        'allow_external': False,
    },

}

In [ ]:
def load_data_gamification():
    # Usamos sep=None y engine='python' para detectar el delimitador automáticamente (comma, semicolon, etc.)
    read_params = {'sep': None, 'engine': 'python', 'on_bad_lines': 'warn'}

    dynamics = pd.read_csv("dataset/dynamicsGam.csv", **read_params)
    type_dynamics = pd.read_csv("dataset/typeDynamicsGam.csv", **read_params)
    elements = pd.read_csv("dataset/elementsGam.csv", **read_params)
    mechanics = pd.read_csv("dataset/mechanicsGam.csv", **read_params)
    narratives = pd.read_csv("dataset/narrativesGam.csv", **read_params)

    dynamics = dynamics.merge(type_dynamics, on="typeId", how="left")
    dynamics_df = pd.DataFrame({
        "item_id": dynamics["dynamicId"].astype(str),
        "item_type": "dynamic",
        "title": dynamics["dynamicGam"],
        "description": dynamics.apply(
            lambda r: f"Dynamic: {r['dynamicGam']}. Type: {r.get('dynamicType', '')}",
            axis=1
        ),
        "type_id": dynamics["typeId"],
        "type_label": dynamics["dynamicType"]
    })

    elements_df = pd.DataFrame({
        "item_id": elements["elementId"].astype(str),
        "item_type": "element",
        "title": elements["elementGam"],
        "description": elements["elementGam"],
        "type_id": None,
        "type_label": None
    })

    mechanics_df = pd.DataFrame({
        "item_id": mechanics["mechanicId"].astype(str),
        "item_type": "mechanic",
        "title": mechanics["mechanicGam"],
        "description": mechanics["mechanicGam"],
        "type_id": None,
        "type_label": None
    })

    narratives_df = pd.DataFrame({
        "item_id": narratives["narrativeId"].astype(str),
        "item_type": "narrative",
        "title": narratives["narrativeGamTitle"],
        "description": narratives["narrativeGamTitle"],
        "type_id": None,
        "type_label": None
    })

    df = pd.concat([dynamics_df, elements_df, mechanics_df, narratives_df], ignore_index=True)

    df["item_id"] = df["item_id"].astype(str)
    df["global_item_id"] = df["item_type"] + "_" + df["item_id"]

    df["full_text"] = (
        "Type: " + df["item_type"].fillna("") +
        ". Title: " + df["title"].fillna("") +
        ". Description: " + df["description"].fillna("") +
        ". Category: " + df["type_label"].fillna("")
    )

    print("\n--- DataFrame Head ---")
    display(df.head())
    print("\n--- DataFrame Info ---")
    df.info()

    return df

In [ ]:
def load_serious_games_ratings():
    try:
        # Load intervention files
        elements_i = pd.read_csv("dataset/elementsIntervention.csv").rename(columns=lambda x: x.strip())
        dynamics_i = pd.read_csv("dataset/dynamicsIntervention.csv").rename(columns=lambda x: x.strip())
        mechanics_i = pd.read_csv("dataset/mechanicsIntervention.csv").rename(columns=lambda x: x.strip())
        narratives_i = pd.read_csv("dataset/narrativesIntervention.csv").rename(columns=lambda x: x.strip())

        # Standardize ID column names to 'item_id'
        elements_i = elements_i.rename(columns={"elementId": "item_id"})
        elements_i["item_type"] = "element"

        dynamics_i = dynamics_i.rename(columns={"dynamicId": "item_id"})
        dynamics_i["item_type"] = "dynamic"

        mechanics_i = mechanics_i.rename(columns={"mechanicId": "item_id"})
        mechanics_i["item_type"] = "mechanic"

        narratives_i = narratives_i.rename(columns={"narrativeId": "item_id"})
        narratives_i["item_type"] = "narrative"

        # Concatenate into a single ratings dataframe
        ratings_df = pd.concat(
            [elements_i, dynamics_i, mechanics_i, narratives_i],
            ignore_index=True
        )

        # Ensure item_id exists after concatenation
        if 'item_id' not in ratings_df.columns:
            id_cols = [c for c in ratings_df.columns if 'Id' in c]
            if id_cols:
                ratings_df['item_id'] = ratings_df[id_cols[0]]
            else:
                raise KeyError("No valid ID column found in intervention files.")

        # Normalizar tipos
        ratings_df["item_id"] = ratings_df["item_id"].astype(str).str.strip()
        ratings_df["item_type"] = ratings_df["item_type"].astype(str).str.strip().str.lower()

        # Crear llave compuesta global
        ratings_df["global_item_id"] = ratings_df["item_type"] + "_" + ratings_df["item_id"]

        # Validaciones opcionales útiles
        if "userId" not in ratings_df.columns:
            raise KeyError("Column 'userId' not found in intervention files.")
        if "rating" not in ratings_df.columns:
            raise KeyError("Column 'rating' not found in intervention files.")

        print("\n--- Ratings DataFrame Head ---")
        display(ratings_df.head())
        print("\n--- Ratings DataFrame Info ---")
        ratings_df.info()

        return ratings_df
    except Exception as e:
        print(f"   ☑′ Using fallback synthetic ratings due to: {e}")
        # Return dummy data to ensure the FCD component doesn't crash the pipeline
        return pd.DataFrame({'userId': [1]*5, 'item_id': [1,2,3,4,5], 'item_type': ['dynamic']*5, 'rating': [5,4,5,4,5], 'global_item_id': ['dynamic_1','dynamic_2','dynamic_3','dynamic_4','dynamic_5']})

In [ ]:
def setup_colab_chromadb():
    """
    Setup ChromaDB en Google Colab para Serious Games:
    1. Monta Google Drive
    2. Configura los directorios locales para persistencia
    """
    #if not is_colab():
    #    print("⚠️ No estamos en Google Colab, saltando setup")
    #    return False

    print("=" * 80)
    print("🚀 CONFIGURACIÓN GOOGLE COLAB - SERIOUS GAMES")
    print("=" * 80)
    print()

    # 1. Montar Google Drive
    print("📁 Montando Google Drive...")
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        print("   ✅ Drive montado en /content/drive\n")
    except Exception as e:
        print(f"   ❌ Error montando Drive: {e}")
        return False

    # 2. Configurar Directorio Local de Chroma para Juegos Serios
    local_path = "/content/chroma_db_serious_games"
    if not os.path.exists(local_path):
        os.makedirs(local_path)
        print(f"   ✅ Directorio creado: {local_path}")
    else:
        print(f"   ℹ️ Directorio existente: {local_path}")

    print("=" * 80)
    print("✅ SETUP COMPLETADO")
    print("=" * 80)
    return True

In [ ]:
# ═══════════════════════════════════════════════════════
# SETUP: CARGAR DATASETS DESDE DRIVE (SERIOUS GAMES ONLY)
# ═══════════════════════════════════════════════════════

# Cache global de colecciones ChromaDB
_chroma_collections = {}
_chroma_client = None

print("=" * 80)
print("🚀 SETUP DE DATASETS - SERIOUS GAMES")
print("=" * 80)
print()

# Detectar entorno
try:
    from google.colab import drive
    IS_COLAB = True
    print("☁️ Entorno: Google Colab")
except ImportError:
    IS_COLAB = False
    print("💻 Entorno: Local")

# Configuración de Paths para Serious Games
DRIVE_BASE = '/content/drive/MyDrive/sistema-recomendador'
DATASET_PATHS = {
    'serious_games': '/content/dataset'  # Carpeta donde residen los CSVs de intervención y gamificación
}

if IS_COLAB:
    print("📁 Verificando montaje de Google Drive...")
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
        print("   ✅ Drive montado")
    else:
        print("   ✅ Drive ya está montado")

    # Verificar existencia de la carpeta de datos en Drive
    serious_games_source = DRIVE_BASE
    if os.path.exists(serious_games_source):
        print(f"   ✅ Carpeta de Serious Games encontrada en Drive: {serious_games_source}")
        # Crear link simbólico o asegurar que la carpeta 'dataset' local existe
        if not os.path.exists('/content/dataset'):
            print("   📦 Vinculando carpeta de datos...")
            # En lugar de copiar, podemos trabajar directamente o copiar los archivos específicos
            # Aquí asumimos que el usuario tiene los archivos en DRIVE_BASE/dataset
            source_path = os.path.join(DRIVE_BASE, 'dataset')
            if os.path.exists(source_path):
                shutil.copytree(source_path, '/content/dataset', dirs_exist_ok=True)
                print("   ✅ Archivos copiados a /content/dataset")
            else:
                print(f"   ⚠️ No se encontró la subcarpeta 'dataset' en {DRIVE_BASE}")
    else:
        print(f"   ❌ No se encontró la base en Drive: {serious_games_source}")

print("\n" + "=" * 80)
print("✅ SETUP FINALIZADO")
print("=" * 80)
print()

🚀 SETUP DE DATASETS - SERIOUS GAMES

☁️ Entorno: Google Colab
📁 Verificando montaje de Google Drive...
   ✅ Drive ya está montado
   ✅ Carpeta de Serious Games encontrada en Drive: /content/drive/MyDrive/sistema-recomendador

✅ SETUP FINALIZADO



### 🔄 Orquestador de Colecciones (Carga Perezosa)

Esta función actúa como un **wrapper de alto nivel** que gestiona el ciclo de vida de las colecciones en memoria:

1.  **Verificación de Cache:** Revisa si la colección ya está cargada en la variable global `_chroma_collections`.
2.  **Generación Bajo Demanda:** Si la colección no existe, coordina la carga del DataFrame, la generación de embeddings y la indexación en ChromaDB.
3.  **Persistencia:** Retorna el objeto de colección listo para realizar consultas (queries).

In [ ]:
def load_data(dataset_name, generate_embeddings=True, show_progress=False):
    """
    Carga dataset y opcionalmente genera embeddings usando sentence-transformers.
    """
    # 1. Caso especial: Serious Games (Carga desde múltiples archivos en directorio)
    if dataset_name == 'serious_games':
        print(f"\n🔄 Cargando dataset '{dataset_name}' (Modo Multi-Archivo)... ")
        df = load_data_gamification()
        embeddings = None
        if generate_embeddings:
            print(f"   🔄 Generando embeddings para {len(df)} items...")
            embeddings = encoder.encode(df['full_text'].tolist(), show_progress_bar=True)
        return df, embeddings

    # 2. Caso estándar: Carga desde archivo único (Legacy)
    config = DATASETS.get(dataset_name)
    if not config:
        raise ValueError(f"Configuración no encontrada para el dataset: {dataset_name}")

    csv_path = DATASET_PATHS.get(dataset_name, config.get('file'))
    if not csv_path or not os.path.exists(csv_path):
        raise FileNotFoundError(f"Dataset CSV no encontrado: {csv_path}")

    print(f"\n🔄 Cargando dataset '{dataset_name}' desde archivo único...")
    df = pd.read_csv(csv_path)
    df = df.rename(columns={
        config['columns']['id']: 'id',
        config['columns']['title']: 'title',
        config['columns']['description']: 'description'
    })

    texts = (df['title'].fillna('') + '. ' + df['description'].fillna('')).tolist()

    if not generate_embeddings:
        return df, None

    embeddings = encoder.encode(texts, show_progress_bar=show_progress)
    return df, embeddings

### 🛠️ Setup: Generación de Vector DB desde CSV (Raw Data)

Esta función es una utilidad de **construcción desde cero**. Se utiliza cuando no se dispone de los índices pre-calculados (los archivos `.tar.gz` en Drive) o cuando se desea regenerar la base de datos con nuevos datos.

**Flujo de trabajo:**
1.  **Ingestión:** Lee los archivos CSV crudos (Movies/Spotify) desde las rutas configuradas.
2.  **Vectorización:** Genera los embeddings para cada item (puede tomar varios minutos).
3.  **Indexación:** Guarda los vectores en ChromaDB de forma persistente en el disco local o de Colab.
4.  **Validación:** Reporta el éxito o fallo de cada dataset.

In [ ]:
def load_or_create_collection(dataset_name, force_recreate=False):
    """
    Carga colección de ChromaDB desde cache o la crea si no existe.

    Args:
        dataset_name (str): 'movies' o 'spotify'
        force_recreate (bool): Si True, fuerza recreación de colección

    Returns:
        chromadb.Collection: Colección lista para queries
    """
    collection_key = f"{dataset_name}_collection"

    if collection_key not in _chroma_collections or force_recreate:
        # Cargar datos y generar embeddings (con barra de progreso para datasets grandes)
        df, embeddings = load_data(dataset_name, generate_embeddings=True, show_progress=True)

        # Inicializar ChromaDB
        _chroma_collections[collection_key] = initialize_vector_db(
            df, embeddings, dataset_name, force_recreate=force_recreate
        )

    return _chroma_collections[collection_key]



In [ ]:
def setup_chromadb_from_csv(dataset_paths, force_recreate=False):
    global DATASET_PATHS

    print("=" * 80)
    print("🚀 SETUP CHROMADB DESDE CSV")
    print("=" * 80)
    print()

    DATASET_PATHS.update(dataset_paths)
    print("📂 Dataset paths configurados:")
    for name, path in dataset_paths.items():
        exists = "✅" if os.path.exists(path) else "❌"
        print(f"   {exists} {name}: {path}")
    print()

    status = {}

    for dataset_name in dataset_paths.keys():
        print("=" * 80)
        print(f"🔧 Procesando dataset: {dataset_name.upper()}")
        print("=" * 80)

        try:
            db_path = "/content/chroma_db_serious_games"
            collection_exists = False

            if os.path.exists(db_path) and not force_recreate:
                try:
                    client = get_chromadb_client(dataset_name)
                    collection = client.get_collection(name=f"{dataset_name}_collection")
                    count = collection.count()
                    if count > 0:
                        collection_exists = True
                        print(f"✅ ChromaDB ya existe con {count} items")
                        status[dataset_name] = True
                        continue
                except: pass

            print(f"📥 Cargando datos y generando embeddings...")
            collection = load_or_create_collection(dataset_name, force_recreate=force_recreate)

            count = collection.count()
            print(f"\n✅ ChromaDB generado exitosamente ({count} items)")
            status[dataset_name] = True

        except Exception as e:
            print(f"❌ Error procesando {dataset_name}: {e}")
            status[dataset_name] = False

    return status

### ⚡ Core del Motor RAG: Optimizaciones, Cliente y Búsqueda

Este bloque contiene la lógica central de recuperación y optimización de rendimiento del sistema. Se divide en tres componentes clave:

#### 1. Sistema de Caché (LRU & Hashing)
Implementa cachés en memoria para **Queries** y **Items**.
* **Objetivo:** Evitar recalcular embeddings para consultas repetidas o items populares durante la re-clasificación (MMR).
* **Impacto:** Reduce drásticamente la latencia en producción.

#### 2. Gestión del Cliente ChromaDB (Singleton)
Maneja la conexión a la base de datos vectorial, abstrayendo si el entorno es **Local**, **Google Colab** o **Cloud**. Asegura que solo exista una instancia del cliente por sesión.

#### 3. Búsqueda Vectorial (Retrieval) y Tuning HNSW
Define las funciones de búsqueda (`rag_retrieval`) y la creación de índices con parámetros **HNSW** (*Hierarchical Navigable Small World*) ajustados para datasets de >100k items:
* `hnsw:search_ef=100`: Aumenta la precisión de búsqueda (recall) a costa de un ligero incremento en latencia.
* `hnsw:M=32`: Mejora la navegabilidad del grafo vectorial.

In [ ]:
setup_colab_chromadb()

🚀 CONFIGURACIÓN GOOGLE COLAB - SERIOUS GAMES

📁 Montando Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   ✅ Drive montado en /content/drive

   ℹ️ Directorio existente: /content/chroma_db_serious_games
✅ SETUP COMPLETADO


True

In [ ]:
# ═══════════════════════════════════════════════════════
# OPTIMIZATION: Item Embedding Cache (para MMR)
# ═══════════════════════════════════════════════════════
_item_embedding_cache = {}
_ITEM_CACHE_MAX_SIZE = 5000

def get_query_embedding_cached(query):
    global _query_embedding_cache
    query_hash = hashlib.md5(query.encode()).hexdigest()
    if query_hash not in _query_embedding_cache:
        _query_embedding_cache[query_hash] = encoder.encode([query])[0]
        if len(_query_embedding_cache) > _CACHE_MAX_SIZE:
            _query_embedding_cache.pop(next(iter(_query_embedding_cache)))
    return _query_embedding_cache[query_hash]

def get_item_embedding_cached(title, description):
    global _item_embedding_cache
    cache_key = f"{title}|{description}"
    key_hash = hashlib.md5(cache_key.encode()).hexdigest()
    if key_hash not in _item_embedding_cache:
        item_text = f"{title}. {description}"
        _item_embedding_cache[key_hash] = encoder.encode([item_text])[0]
        if len(_item_embedding_cache) > _ITEM_CACHE_MAX_SIZE:
            _item_embedding_cache.pop(next(iter(_item_embedding_cache)))
    return _item_embedding_cache[key_hash]

def get_chromadb_client(dataset_name='serious_games'):
    """
    Obtiene cliente de ChromaDB persistente para Serious Games.
    """
    global _chroma_client
    db_path = "/content/chroma_db_serious_games"
    client_key = "serious_games_client"

    if client_key not in _chroma_collections:
        print(f"📱 Inicializando ChromaDB para Serious Games...")
        if not os.path.exists(db_path):
            os.makedirs(db_path)
        _chroma_collections[client_key] = chromadb.PersistentClient(path=db_path)
        print(f"   ✅ ChromaDB inicializado en {db_path}")

    return _chroma_collections[client_key]

def initialize_vector_db(df, embeddings, dataset_name='serious_games', force_recreate=False):
    client = get_chromadb_client()
    collection_name = "serious_games_collection"

    if force_recreate:
        try:
            client.delete_collection(name=collection_name)
            print(f"   🗑️ Colección existente eliminada")
        except: pass

    try:
        collection = client.get_collection(name=collection_name)
        if collection.count() > 0:
            print(f"   ✅ Colección cargada ({collection.count()} items)")
            return collection
    except:
        print(f"   📦 Creando nueva colección HNSW para Serious Games...")
        collection = client.create_collection(
            name=collection_name,
            metadata={"hnsw:search_ef": 100, "hnsw:M": 32, "hnsw:space": "cosine"}
        )

    batch_size = 5000
    total_items = len(df)
    for i in range(0, total_items, batch_size):
        batch_end = min(i + batch_size, total_items)
        batch_df = df.iloc[i:batch_end]
        batch_embeddings = embeddings[i:batch_end]
        ids = [f"sg_{idx}" for idx in batch_df.index]
        documents = batch_df['title'].fillna('').astype(str).tolist()
        metadatas = [{'id': str(row['item_id']), 'description': str(row['description'])} for _, row in batch_df.iterrows()]

        collection.add(embeddings=batch_embeddings.tolist(), documents=documents, metadatas=metadatas, ids=ids)
        print(f"      Batch {i//batch_size + 1}: {batch_end - i} items agregados")

    return collection

def rag_retrieval(query, collection, top_k=30):
    query_emb = get_query_embedding_cached(query)
    results = collection.query(query_embeddings=[query_emb.tolist()], n_results=top_k, include=['documents', 'metadatas', 'distances'])

    if not results['ids'] or len(results['ids'][0]) == 0:
        return pd.DataFrame(columns=['id', 'title', 'description', 'similarity'])

    return pd.DataFrame({
        'id': [m['id'] for m in results['metadatas'][0]],
        'title': results['documents'][0],
        'description': [m['description'] for m in results['metadatas'][0]],
        'similarity': [1 - d for d in results['distances'][0]]
    })

In [ ]:
status = setup_chromadb_from_csv({'serious_games': DATASET_PATHS['serious_games']})

if status.get('serious_games'):
    print("\n🎉 ChromaDB de Serious Games listo para uso!")
    print("   Ahora puedes ejecutar el motor de recomendación.")
else:
    print("\n⚠️ Hubo un problema al inicializar el dataset de Serious Games. Revisa los mensajes arriba.")

🚀 SETUP CHROMADB DESDE CSV

📂 Dataset paths configurados:
   ✅ serious_games: /content/dataset

🔧 Procesando dataset: SERIOUS_GAMES
📱 Inicializando ChromaDB para Serious Games...
   ✅ ChromaDB inicializado en /content/chroma_db_serious_games
✅ ChromaDB ya existe con 108 items

🎉 ChromaDB de Serious Games listo para uso!
   Ahora puedes ejecutar el motor de recomendación.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:


# Optimización: RapidFuzz
try:
    from rapidfuzz import process, fuzz
    USE_RAPIDFUZZ = True
    print("✅ RapidFuzz detectado (Optimización activada)")
except ImportError:
    USE_RAPIDFUZZ = False
    from difflib import SequenceMatcher
    print("⚠️ RapidFuzz no encontrado. Usando difflib (más lento).")

# Configuración de Logs
VERBOSE = True
TOP_K_CANDIDATES = 200
USE_VECTORIZED_MMR = True
print("✅ Importaciones completadas")

✅ RapidFuzz detectado (Optimización activada)
✅ Importaciones completadas


### Configuración de API Keys
Define aquí tus claves. El sistema detectará automáticamente cuáles están disponibles.

In [ ]:
# ⬇️ INGRESA TUS API KEYS AQUÍ ⬇️
from google.colab import userdata
try:
    # Intenta cargar desde los Secretos de Colab
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')

    print("✅ Claves cargadas desde Colab Secrets")
except ImportError:
    # Fallback para entorno local (ver Opción 2)
    print("⚠️ No estamos en Colab o no se encontraron secretos.")


# Carga automática desde .env si existe
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

print("Variables de entorno configuradas.")

✅ Claves cargadas desde Colab Secrets
Variables de entorno configuradas.


<cell_type>markdown</cell_type>## 4. Componente RAG - Semantic Search
Implementación de búsqueda semántica directa usando cosine similarity sobre embeddings.

## 2. Arquitectura de Proveedores LLM (Model-Agnostic)
Esta capa de abstracción permite que el sistema no dependa de un proveedor específico.

In [ ]:
class BaseLLMProvider:
    def __init__(self, model_name, **kwargs):
        self.model_name = model_name
        self.config = kwargs
    def generate(self, prompt, max_tokens=1000, temperature=0.7):
        raise NotImplementedError("Subclass must implement generate()")

class OpenAIProvider(BaseLLMProvider):
    def __init__(self, model_name, **kwargs):
        super().__init__(model_name, **kwargs)
        self.client = openai.OpenAI(
            api_key=kwargs.get('api_key'),
            base_url=kwargs.get('base_url')
        )
    def generate(self, prompt, max_tokens=1000, temperature=0.7):
        response = self.client.chat.completions.create(
            model=self.model_name,
            messages=[{'role': 'user', 'content': prompt}],
            max_tokens=max_tokens,
            temperature=temperature
        )
        print(response.choices[0].message.content)
        return response.choices[0].message.content

class AnthropicProvider(BaseLLMProvider):
    def __init__(self, model_name, **kwargs):
        super().__init__(model_name, **kwargs)
        self.client = anthropic.Anthropic(api_key=kwargs.get('api_key'))
    def generate(self, prompt, max_tokens=1000, temperature=0.7):
        response = self.client.messages.create(
            model=self.model_name,
            max_tokens=max_tokens,
            messages=[{'role': 'user', 'content': prompt}],
            temperature=temperature
        )
        print(response.content[0].text)
        return response.content[0].text

import re

class GoogleProvider(BaseLLMProvider):
    def __init__(self, model_name, **kwargs):
        super().__init__(model_name, **kwargs)
        # Inicialización de la nueva SDK google-genai
        self.client = genai.Client(api_key=kwargs.get('api_key'))
        self.model_id = model_name

    def generate(self, prompt, max_tokens=8192, temperature=0.1):
        """
        Genera contenido asegurando un formato JSON válido.
        Optimizado para evitar el truncamiento por 'Thinking process'.
        """
        # Forzamos un límite alto para evitar el corte prematuro
        effective_tokens = max(max_tokens, 8192)

        try:
            # Usamos GenerateContentConfig para asegurar que la SDK aplique los parámetros
            response = self.client.models.generate_content(
                model=self.model_id,
                contents=prompt,
                config=types.GenerateContentConfig(
                    max_output_tokens=effective_tokens,
                    temperature=temperature,
                    response_mime_type='application/json',
                    # Añadimos instrucción de sistema para ahorrar tokens de razonamiento
                    system_instruction="Eres un exportador de datos JSON puro. No expliques, no razones, no pienses en voz alta. Genera directamente el JSON solicitado."
                )
            )

            # Verificación de por qué terminó el modelo
            if response.candidates and response.candidates[0].finish_reason == 'MAX_TOKENS':
                print(f"⚠️ CRÍTICO: El modelo agotó los {effective_tokens} tokens. El JSON estará incompleto.")

            text = response.text
            if not text:
                return '[]'

            # Limpieza de posibles bloques Markdown residuales
            clean_text = re.sub(r'```json\s?|```', '', text).strip()

            # Si el texto es muy corto y no empieza con estructura JSON, algo salió mal
            if not (clean_text.startswith('{') or clean_text.startswith('[')):
                 return f'{{"error": "Respuesta inválida", "raw": "{clean_text[:100]}"}}'

            return clean_text

        except Exception as e:
            error_msg = str(e).replace('"', "'")
            return f'{{"error": "Fallo en Gemini: {error_msg}"}}'
PROVIDER_REGISTRY = {
    'openai': OpenAIProvider,
    'anthropic': AnthropicProvider,
    'google': GoogleProvider,
}
def create_llm_provider(provider_type, model_name, **kwargs):
    if provider_type not in PROVIDER_REGISTRY:
        raise ValueError(f"Proveedor desconocido: {provider_type}")
    return PROVIDER_REGISTRY[provider_type](model_name, **kwargs)

## 3. Configuración Global
Definimos los modelos a usar, los parámetros de los datasets y los prompts detallados del sistema.

In [ ]:
# --- Configuración de LLMs ---
LLMS = {

    'gemini': {
        'enabled': bool(os.getenv('GOOGLE_API_KEY')),
        'provider': 'google',
        # Updated to a stable version as 2.0-flash returned a 404
        'model': 'gemini-2.5-flash',
        'api_key': os.getenv('GOOGLE_API_KEY'),
    },
    'groq_llama': {
        'enabled': bool(os.getenv('GROQ_API_KEY')),
        'provider': 'openai',
        'model': 'llama-3.3-70b-versatile',
        'api_key': os.getenv('GROQ_API_KEY'),
        'base_url': 'https://api.groq.com/openai/v1',
    },
    'claude_haiku': {
        'enabled': bool(os.getenv('ANTHROPIC_API_KEY')),
        'provider': 'anthropic',
        'model': 'claude-haiku-4-5',
        'api_key': os.getenv('ANTHROPIC_API_KEY'),
    },
    'gpt4o_mini': {
        'enabled': bool(os.getenv('OPENAI_API_KEY')),
        'provider': 'openai',
        'model': 'gpt-4o-mini',
        'api_key': os.getenv('OPENAI_API_KEY'),
    }
}

ACTIVE_LLMS = {k: v for k, v in LLMS.items() if v['enabled']}
print(f"LLMs Activos: {list(ACTIVE_LLMS.keys())}")

LLMs Activos: ['gemini', 'claude_haiku', 'gpt4o_mini']


### Prompts Template

In [ ]:
SERIOUS_GAMES_PROMPT = """You are an expert recommendation system for gamified serious games.

Available components:
{context}

User query:
{query}

Instructions:
1. Recommend exactly ONE serious game design based on the components above.
2. Keep the 'justification' and 'description' concise (under 3 sentences).
3. Ensure the JSON is properly formatted.

Output format:
[
  {{
    "recommendation": "Name",
    "ranking": 0.95,
    "description": "Concise explanation",
    "source": "database",
    "sector": "Sector",
    "context": "Context",
    "dynamics": {{
      "progression": "Description",
      "emotion": "Description",
      "cognitive": "Description",
      "motivation": "Description"
    }},
    "mechanics": ["M1", "M2"],
    "elements": ["E1", "E2"],
    "narrative": "Story",
    "justification": "Alignment reasoning"
  }}
]
"""

PROMPTS = {
    'serious_games': SERIOUS_GAMES_PROMPT,
}
print("✅ SERIOUS_GAMES_PROMPT fixed with escaped braces for JSON formatting.")

✅ SERIOUS_GAMES_PROMPT fixed with escaped braces for JSON formatting.


### 🧠 Módulo de Filtrado Colaborativo (FCD)

Este bloque encapsula toda la lógica "social" del sistema de recomendación. A diferencia del enfoque basado en contenido (RAG/Embeddings) que analiza *qué es* el ítem, este módulo analiza *quién interactúa* con el ítem.

**Funciones Clave:**

1.  **Generación Sintética (Spotify):** Crea un dataset realista de usuarios y ratings (1-5) simulando sesgos de popularidad y preferencias de género, necesario ante la falta de datos explícitos públicos.
2.  **Optimización Matricial:** Convierte los logs de interacción en una **Matriz Dispersa (Sparse Matrix)** para optimizar cálculos algebraicos.
3.  **Algoritmo User-Based CF:**
    * Calcula la **Similitud del Coseno** entre vectores de usuarios.
    * Identifica vecinos con gustos similares.
    * Recomienda ítems que esos vecinos valoraron positivamente (Rating $\ge$ 4.0) y que el usuario actual aún no conoce.

In [ ]:
def generate_synthetic_ratings_spotify(n_users=500, output_file='synthetic_spotify_ratings.csv'):
    """
    Genera ratings sintéticos para Spotify con patrones realistas.

    Estrategia:
    1. Usuarios tienen "perfiles de gusto" basados en géneros
    2. Ratings siguen distribución normal (μ=3.5, σ=1.2)
    3. Usuarios activos (20%) tienen más ratings
    4. Usuarios nuevos (cold start - 10%) tienen < 10 ratings

    Args:
        n_users: número de usuarios sintéticos
        output_file: archivo CSV de salida

    Returns:
        DataFrame con ratings generados
    """
    import random
    from datetime import datetime, timedelta

    print(f"\n🔧 Generando ratings sintéticos para Spotify...")
    print(f"   Usuarios: {n_users}")

    # Para generar ratings sintéticos necesitamos IDs de tracks
    # Como no tenemos dataset de Spotify, generaremos IDs sintéticos
    n_tracks = 10000  # Simular 10K tracks
    track_ids = [f"track_{i:05d}" for i in range(n_tracks)]

    # Definir perfiles de usuario (géneros preferidos)
    genres = ['pop', 'rock', 'electronic', 'hip-hop', 'jazz', 'classical',
              'country', 'r&b', 'indie', 'metal']

    ratings_data = []

    for user_id in range(1, n_users + 1):
        # Determinar tipo de usuario
        user_type = random.random()

        if user_type < 0.10:
            # 10% usuarios cold start (< 10 ratings)
            n_ratings = random.randint(2, 9)
        elif user_type < 0.30:
            # 20% usuarios muy activos (100-200 ratings)
            n_ratings = random.randint(100, 200)
        else:
            # 70% usuarios normales (20-80 ratings)
            n_ratings = random.randint(20, 80)

        # Perfil del usuario: 2-3 géneros favoritos
        user_genres = random.sample(genres, k=random.randint(2, 3))

        # Generar ratings
        base_timestamp = int((datetime.now() - timedelta(days=365)).timestamp())

        for _ in range(n_ratings):
            track_id = random.choice(track_ids)

            # Rating basado en preferencias (simulado)
            # Si el track es del género favorito, rating más alto
            is_favorite_genre = random.random() < 0.6  # 60% de género favorito

            if is_favorite_genre:
                rating = max(1.0, min(5.0, random.gauss(4.2, 0.8)))
            else:
                rating = max(1.0, min(5.0, random.gauss(3.0, 1.2)))

            # Redondear a 0.5
            rating = round(rating * 2) / 2

            # Timestamp aleatorio en el último año
            timestamp = base_timestamp + random.randint(0, 365 * 24 * 3600)

            ratings_data.append({
                'userId': user_id,
                'track_id': track_id,
                'rating': rating,
                'timestamp': timestamp
            })

    ratings_df = pd.DataFrame(ratings_data)

    # Eliminar duplicados (mismo usuario, mismo track)
    ratings_df = ratings_df.drop_duplicates(subset=['userId', 'track_id'], keep='first')

    # Guardar
    ratings_df.to_csv(output_file, index=False)

    print(f"   ✅ Generados: {len(ratings_df):,} ratings")
    print(f"   ✅ Guardado en: {output_file}")
    print(f"   📊 Distribución:")
    print(f"      - Rating promedio: {ratings_df['rating'].mean():.2f}")
    print(f"      - Rating std: {ratings_df['rating'].std():.2f}")
    print(f"      - Ratings/usuario: {len(ratings_df)/n_users:.1f} promedio")

    return ratings_df


def load_user_ratings(dataset_name):
    """
    Carga ratings de usuarios y crea matriz dispersa para FCD.

    Args:
        dataset_name: 'movies' o 'spotify'

    Returns:
        tuple: (ratings_df, user_item_matrix, user_id_map, item_id_map)
            - ratings_df: DataFrame original
            - user_item_matrix: scipy.sparse.csr_matrix (n_users × n_items)
            - user_id_map: dict {original_user_id: matrix_index}
            - item_id_map: dict {original_item_id: matrix_index}
    """
    from scipy.sparse import csr_matrix
    if dataset_name == 'serious_games':
        ratings_df = load_serious_games_ratings()
        item_col = 'global_item_id'
    else:
        raise ValueError(f"Dataset '{dataset_name}' no soportado")

    print(f"   ✅ Cargados: {len(ratings_df):,} ratings")
    print(f"   📊 Usuarios: {ratings_df['userId'].nunique():,}")
    print(f"   📊 Items: {ratings_df[item_col].nunique():,}")
    print(f"   📊 Sparsity: {(1 - len(ratings_df) / (ratings_df['userId'].nunique() * ratings_df[item_col].nunique())) * 100:.2f}%")

    # Crear mapeos de IDs a índices de matriz
    unique_users = sorted(ratings_df['userId'].unique())
    unique_items = sorted(ratings_df[item_col].unique())

    user_id_map = {user_id: idx for idx, user_id in enumerate(unique_users)}
    item_id_map = {item_id: idx for idx, item_id in enumerate(unique_items)}

    # Crear matriz dispersa
    row_indices = [user_id_map[uid] for uid in ratings_df['userId']]
    col_indices = [item_id_map[iid] for iid in ratings_df[item_col]]
    ratings_values = ratings_df['rating'].values

    n_users = len(unique_users)
    n_items = len(unique_items)

    user_item_matrix = csr_matrix(
        (ratings_values, (row_indices, col_indices)),
        shape=(n_users, n_items)
    )

    print(f"   ✅ Matriz creada: {user_item_matrix.shape} (sparse)")

    return ratings_df, user_item_matrix, user_id_map, item_id_map


def calculate_user_similarity(user_item_matrix, user_idx, top_k=50):
    """
    Calcula similitud entre usuarios usando correlación de coseno.

    Args:
        user_item_matrix: scipy.sparse matriz (n_users × n_items)
        user_idx: índice del usuario target en la matriz
        top_k: número de usuarios similares a retornar

    Returns:
        list: [(similar_user_idx, similarity_score), ...] ordenado por similitud desc
    """
    from sklearn.metrics.pairwise import cosine_similarity

    # Obtener vector del usuario target
    user_vector = user_item_matrix[user_idx]

    # Calcular similitud con todos los usuarios
    similarities = cosine_similarity(user_vector, user_item_matrix).flatten()

    # Excluir al mismo usuario
    similarities[user_idx] = -1

    # Obtener top-K más similares
    top_indices = np.argsort(similarities)[-top_k:][::-1]

    # Filtrar usuarios con similitud > 0
    similar_users = [
        (idx, similarities[idx])
        for idx in top_indices
        if similarities[idx] > 0
    ]

    return similar_users


def fcd_recommendations(user_id, dataset_name, ratings_df, user_item_matrix,
                        user_id_map, item_id_map, top_k=30):
    """
    Filtrado Colaborativo Distribuido (FCD).
    Genera recomendaciones basadas en usuarios similares.

    Algoritmo:
    1. Encontrar usuarios similares (top-50)
    2. Obtener items que ellos calificaron bien (rating ≥ 4.0)
    3. Eliminar items ya consumidos por user_id
    4. Rankear por weighted rating score

    Args:
        user_id: ID original del usuario
        dataset_name: 'movies' o 'spotify'
        ratings_df: DataFrame de ratings
        user_item_matrix: matriz dispersa
        user_id_map: mapeo user_id → matrix_index
        item_id_map: mapeo item_id → matrix_index
        top_k: número de recomendaciones

    Returns:
        list: [{'item_id': ..., 'fcd_score': ..., 'source': 'fcd'}, ...]
    """
    # Verificar que el usuario existe
    if user_id not in user_id_map:
        if VERBOSE:
            print(f"   ⚠️  Usuario {user_id} no encontrado en ratings, retornando lista vacía")
        return []

    user_idx = user_id_map[user_id]

    # 1. Encontrar usuarios similares
    similar_users = calculate_user_similarity(user_item_matrix, user_idx, top_k=50)

    if not similar_users:
        if VERBOSE:
            print(f"   ⚠️  No se encontraron usuarios similares para user {user_id}")
        return []

    if VERBOSE:
        print(f"\n🔍 FCD para usuario {user_id}:")
        print(f"   ✅ {len(similar_users)} usuarios similares encontrados")
        print(f"   📊 Similitud promedio: {np.mean([sim for _, sim in similar_users]):.3f}")

    # 2. Obtener items consumidos por el usuario target
    item_col = 'movieId' if dataset_name == 'movies' else 'track_id'
    user_ratings = ratings_df[ratings_df['userId'] == user_id]
    consumed_items = set(user_ratings[item_col].values)

    # 3. Predecir ratings para items no consumidos
    item_scores = {}

    for similar_user_idx, similarity in similar_users:
        # Obtener items calificados por este usuario similar
        similar_user_items = user_item_matrix[similar_user_idx].nonzero()[1]

        for item_idx in similar_user_items:
            # Obtener ID original del item
            item_id = [iid for iid, idx in item_id_map.items() if idx == item_idx][0]

            # Skip si ya fue consumido por el usuario target
            if item_id in consumed_items:
                continue

            # Obtener rating del usuario similar para este item
            rating = user_item_matrix[similar_user_idx, item_idx]

            # Solo considerar ratings altos (≥ 4.0)
            if rating >= 4.0:
                # Weighted score: similarity × rating
                if item_id not in item_scores:
                    item_scores[item_id] = {'total_score': 0, 'total_sim': 0}

                item_scores[item_id]['total_score'] += similarity * rating
                item_scores[item_id]['total_sim'] += similarity

    # 4. Calcular score final normalizado
    recommendations = []

    for item_id, scores in item_scores.items():
        if scores['total_sim'] > 0:
            # Score ponderado normalizado
            fcd_score = scores['total_score'] / scores['total_sim']
            # Normalizar a [0, 1]: rating está en [1, 5]
            fcd_score_norm = (fcd_score - 1) / 4

            recommendations.append({
                'item_id': str(item_id),
                'fcd_score': fcd_score_norm,
                'source': 'fcd',
                'n_recommendations': int(scores['total_sim'])  # Cuántos usuarios lo recomendaron
            })

    # Ordenar por score descendente
    recommendations.sort(key=lambda x: x['fcd_score'], reverse=True)

    if VERBOSE:
        print(f"   ✅ {len(recommendations)} items candidatos de FCD")
        if recommendations:
            print(f"   📊 Score promedio: {np.mean([r['fcd_score'] for r in recommendations]):.3f}")

    return recommendations[:top_k]



### Componente 1: Balanceo LLM/FCD

Este componente decide cuánto peso (`weight`) dar a las recomendaciones de los LLMs (que entienden la semántica de la consulta) vs. el FCD (que entiende el historial personal del usuario).

**Lógica de Balanceo Adaptativo:**

| Historial del Usuario | Peso LLM | Peso FCD | Justificación |
| :--- | :---: | :---: | :--- |
| 0-9 ratings | 90% | 10% | **Cold Start:** El historial es muy bajo. Confiamos casi totalmente en la semántica (LLM). |
| 10-49 ratings | 70% | 30% | **Intermedio:** El historial empieza a ser útil. El FCD complementa al LLM. |
| 50+ ratings | 60% | 40% | **Veterano:** El FCD tiene un peso significativo, pero el LLM (semántica) sigue dominando. |

El peso del LLM (`weight_llm`) se distribuye equitativamente entre todos los LLMs activos (ej. si hay 5 LLMs y `weight_llm` es 0.70, cada uno tiene un peso de 0.14).

In [ ]:
# ═══════════════════════════════════════════════════════
# COMPONENTE 1: BALANCING (SERIOUS GAMES ONLY)
# ═══════════════════════════════════════════════════════

def calculate_weights(user_id=None, dataset_name='serious_games'):
    """
    Calcula pesos dinámicos para LLMs y FCD según historial del usuario.
    Estrategia de balanceo adaptativo para Serious Games.
    """
    num_llms = len(ACTIVE_LLMS)

    if num_llms == 0:
        raise ValueError("No hay LLMs activos. Configure al menos una API key.")

    # Caso 1: Sin usuario → 100% LLM (Recomendación basada en perfil teórico)
    if user_id is None:
        weight_per_llm = 1.0 / num_llms
        return {
            'llm': 1.0,
            'fcd': 0.0,
            'llm_individual': {llm_name: weight_per_llm for llm_name in ACTIVE_LLMS.keys()}
        }

    # Caso 2: Con usuario → balanceo adaptativo según intervenciones previas
    try:
        # Cargar ratings específicos de Serious Games
        ratings_df = load_serious_games_ratings()
        user_ratings = ratings_df[ratings_df['userId'] == user_id]
        n_ratings = len(user_ratings)
    except:
        n_ratings = 0

    # Calcular pesos según historial de intervención
    if n_ratings < 5:
        # Usuario con pocas intervenciones → Priorizar perfil cognitivo (LLM)
        weight_llm = 0.90
        weight_fcd = 0.10
        user_profile = 'new_user'
    elif n_ratings < 20:
        # Usuario intermedio
        weight_llm = 0.70
        weight_fcd = 0.30
        user_profile = 'active_user'
    else:
        # Usuario veterano → El historial de éxito (FCD) tiene más peso
        weight_llm = 0.60
        weight_fcd = 0.40
        user_profile = 'veteran_user'

    weight_per_llm = weight_llm / num_llms

    if VERBOSE:
        print(f"\n⚖️ Balanceo adaptativo (Serious Games - user {user_id}):")
        print(f"   📊 Intervenciones: {n_ratings} → perfil '{user_profile}'")
        print(f"   📊 Pesos: {weight_llm*100:.0f}% LLM, {weight_fcd*100:.0f}% FCD")

    return {
        'llm': weight_llm,
        'fcd': weight_fcd,
        'llm_individual': {llm_name: weight_per_llm for llm_name in ACTIVE_LLMS.keys()},
        'user_profile': user_profile,
        'n_ratings': n_ratings
    }

### Componente 2: Generación Multi-Fuente

Este componente genera las listas de candidatos desde dos tipos de fuentes en paralelo:

1.  **Filtrado Colaborativo Distribuido (FCD):** (Si `user_id` está presente). Se basa en el algoritmo *User-Based Collaborative Filtering*.
    a.  Carga la matriz de ratings `(usuarios x ítems)`.
    b.  Encuentra los 50 usuarios más similares al `user_id` actual (usando Similitud de Coseno sobre sus vectores de ratings).
    c.  Predice el rating que el usuario actual daría a ítems que no ha visto, basándose en un promedio ponderado de los ratings de esos usuarios similares.
    d.  Retorna los ítems con la predicción de rating más alta.

2.  **Comité de LLMs (vía RAG):**
    a.  **Recuperación (RAG):** Se obtienen los 200 candidatos más relevantes de ChromaDB (`rag_retrieval`).
    b.  **Construcción de Contexto:** Se formatean estos 200 candidatos en un texto legible para el LLM (`build_context`).
    c.  **Consulta Paralela:** Se envía el *mismo* prompt (contexto + query) a *todos* los `ACTIVE_LLMS` en paralelo usando `ThreadPoolExecutor` (`query_all_llms`).
    d.  **Parsing:** Cada respuesta JSON del LLM se parsea y limpia (`parse_llm_json`).

La salida de este componente son dos conjuntos de listas: `llm_results` (un dict con las 5 recomendaciones de cada LLM) y `fcd_results` (una lista con las 30 mejores recomendaciones del FCD).

In [ ]:
# ═══════════════════════════════════════════════════════
# COMPONENTE 2: MULTI-SOURCE GENERATION
# ═══════════════════════════════════════════════════════


def semantic_search(query, embeddings, df, top_k=30):
    """
    Búsqueda semántica usando similitud de coseno.

    Args:
        query (str): Consulta del usuario
        embeddings (np.ndarray): Embeddings de items (n x 384)
        df (pd.DataFrame): Dataset con metadata
        top_k (int): Número de candidatos a retornar

    Returns:
        pd.DataFrame: Top-K items más similares con columna 'similarity'
    """
    # Generar embedding de query
    query_emb = encoder.encode([query])[0]

    # Calcular similitud coseno con todos los items de forma robusta
    # cosine_sim = (A · B) / (||A|| * ||B||)

    # Primero calcular normas para detectar problemas antes del matmul
    norms_embeddings = np.linalg.norm(embeddings, axis=1)
    norm_query = np.linalg.norm(query_emb)

    # Evitar división por cero (epsilon grande para estabilidad numérica)
    epsilon = 1e-8
    norms_embeddings = np.where(norms_embeddings == 0, epsilon, norms_embeddings)
    norm_query = max(norm_query, epsilon)

    # Normalizar vectores antes del dot product (evita overflow/underflow)
    embeddings_normalized = embeddings / norms_embeddings[:, np.newaxis]
    query_normalized = query_emb / norm_query

    # Dot product con vectores normalizados (más estable numéricamente)
    with np.errstate(divide='ignore', over='ignore', under='ignore', invalid='ignore'):
        similarities = embeddings_normalized @ query_normalized

    # Clip para evitar valores fuera de rango por errores numéricos
    similarities = np.clip(similarities, -1.0, 1.0)

    # Normalizar a [0, 1]: (sim + 1) / 2
    similarities_norm = (similarities + 1) / 2

    # Obtener top-K indices
    top_indices = np.argsort(similarities_norm)[-top_k:][::-1]

    # Crear DataFrame con resultados
    results = df.iloc[top_indices].copy()
    results['similarity'] = similarities_norm[top_indices]

    return results.reset_index(drop=True)


def build_context(items, dataset_name):
    """
    Construye contexto enriquecido para prompts de LLMs.

    Args:
        items (pd.DataFrame): Items candidatos (top-30)
        dataset_name (str): 'movies' o 'spotify'

    Returns:
        str: Contexto formateado para insertar en prompt
    """
    lines = []

    if dataset_name == 'movies':
        for i, row in items.iterrows():
            # Formato: [1] 296 - Pulp Fiction (1994) - Crime Drama Thriller
            # Extraer año del título si está presente
            title = row['title']
            item_id = row['id']
            genres = row['description']

            lines.append(f"{i+1}. [{item_id}] {title} - {genres}")

    elif dataset_name == 'spotify':
        for i, row in items.iterrows():
            # Formato: [1] track_id - Song by Artist - genre (dance: 0.X, energy: 0.Y)
            title = row['title']
            item_id = row['id']
            genre = row['description']

            # Agregar audio features si existen
            audio_info = ""
            if 'danceability' in row and 'energy' in row:
                audio_info = f" (dance: {row['danceability']:.2f}, energy: {row['energy']:.2f})"

            artist = row.get('artists', 'Unknown')
            lines.append(f"{i+1}. [{item_id}] {title} by {artist} - {genre}{audio_info}")
    else:
        # Fallback genérico
        for i, row in items.iterrows():
            lines.append(f"{i+1}. [{row['id']}] {row['title']} - {row['description']}")

    return '\n'.join(lines)


def call_llm(llm_name, prompt):
    """
    Llama a un LLM con retry logic y exponential backoff.

    ARQUITECTURA 100% MODEL-AGNOSTIC:
    - Usa el patrón Factory para crear proveedores
    - Cualquier modelo nuevo se agrega solo en LLMS config
    - No requiere modificar este código

    Args:
        llm_name (str): Nombre del LLM ('gpt4o', 'claude', 'gemini', etc.)
        prompt (str): Prompt completo a enviar

    Returns:
        str: Respuesta del LLM (JSON string esperado)
    """
    config = ACTIVE_LLMS[llm_name]

    for attempt in range(RETRY_ATTEMPTS):
        try:
            # Crear provider usando factory pattern (model-agnostic)
            provider_kwargs = {
                'api_key': config.get('api_key'),
                'base_url': config.get('base_url'),  # Solo para OpenAI-compatible
            }

            provider = create_llm_provider(
                provider_type=config['provider'],
                model_name=config['model'],
                **provider_kwargs
            )

            # Generar respuesta (interfaz unificada)
            print()
            return provider.generate(
                prompt=prompt,
                max_tokens=MAX_TOKENS,
                temperature=TEMPERATURE
            )

        except Exception as e:
            if attempt == RETRY_ATTEMPTS - 1:
                # Último intento falló
                error_msg = f"{str(e)[:150]}"
                print(f"   ⚠️ {llm_name}: {error_msg}")

                # Mostrar traceback completo solo en modo verbose
                if VERBOSE:
                    import traceback
                    print(f"      Full error: {traceback.format_exc()[:300]}")
                return ""
            # Exponential backoff: 1s, 2s, 4s
            time.sleep(2 ** attempt)

    return ""


def parse_llm_json(response, llm_name):
    """
    Parsea respuesta JSON del LLM.

    Args:
        response (str): Respuesta del LLM (debe ser JSON array)
        llm_name (str): Nombre del LLM (para logging)

    Returns:
        list: Lista de dicts con {recommendation, ranking, description, source}
              o lista vacía si falla el parsing
    """
    if not response or not response.strip():
        return []

    try:
        # Intentar parsear JSON directamente
        data = json.loads(response)

        # Validar que sea una lista
        if not isinstance(data, list):
            print(f"   ⚠️ {llm_name}: Response no es un array JSON")
            return []

        # Validar que cada item tenga los campos requeridos
        valid_items = []
        for item in data:
            if all(k in item for k in ['recommendation', 'ranking', 'description']):
                valid_items.append({
                    'recommendation': str(item['recommendation']),
                    'ranking': float(item['ranking']),
                    'description': str(item['description']),
                    'source': str(item.get('source', 'database'))  # NUEVO: source (default: database)
                })
            else:
                print(f"   ⚠️ {llm_name}: Item sin campos requeridos: {item}")

        return valid_items

    except json.JSONDecodeError as e:
        # Intentar múltiples estrategias de limpieza

        # Estrategia 1: Extraer solo el JSON array
        try:
            start = response.find('[')
            end = response.rfind(']') + 1
            if start != -1 and end > start:
                cleaned = response[start:end]
                data = json.loads(cleaned)
                if isinstance(data, list):
                    print(f"   ℹ️ {llm_name}: JSON extraído (estrategia 1)")
                    return parse_llm_json(cleaned, llm_name)
        except:
            pass

        # Estrategia 2: Arreglar strings sin terminar
        try:
            # Buscar el error de "unterminated string"
            if "Unterminated string" in str(e):
                # Intentar cerrar strings abiertos agregando comilla antes del cierre
                cleaned = response
                # Agregar comilla de cierre si falta
                if cleaned.count('"') % 2 != 0:
                    # Hay un número impar de comillas - agregar una antes del ]
                    last_bracket = cleaned.rfind(']')
                    if last_bracket > 0:
                        cleaned = cleaned[:last_bracket] + '"' + cleaned[last_bracket:]

                data = json.loads(cleaned)
                if isinstance(data, list):
                    print(f"   ℹ️ {llm_name}: JSON reparado (comilla agregada)")
                    return parse_llm_json(cleaned, llm_name)
        except:
            pass

        # Estrategia 3: Usar regex para extraer objetos válidos
        try:
            import re
            # Buscar objetos JSON individuales
            objects = re.findall(r'\{[^}]*"recommendation"[^}]*"ranking"[^}]*"description"[^}]*\}', response)
            if objects:
                # Intentar parsear cada objeto
                valid_items = []
                for obj_str in objects:
                    try:
                        obj = json.loads(obj_str)
                        if all(k in obj for k in ['recommendation', 'ranking', 'description']):
                            valid_items.append({
                                'recommendation': str(obj['recommendation']),
                                'ranking': float(obj['ranking']),
                                'description': str(obj['description']),
                                'source': str(obj.get('source', 'database'))  # NUEVO
                            })
                    except:
                        continue

                if valid_items:
                    print(f"   ℹ️ {llm_name}: {len(valid_items)} items extraídos (regex) - esperaba 5")
                    if VERBOSE and len(valid_items) < 5:
                        print(f"      JSON raw (primeros 500 chars): {response[:500]}")
                    return valid_items
        except:
            pass

        print(f"   ⚠️ {llm_name}: JSON inválido - {str(e)[:150]}")
        if VERBOSE:
            print(f"      JSON raw (primeros 500 chars): {response[:500]}")
        return []


def query_all_llms(query, candidates, dataset_name):
    """
    Consulta todos los LLMs activos (en paralelo si USE_PARALLEL_LLMS=True).

    OPTIMIZATION: ThreadPoolExecutor permite 3-5x speedup vs secuencial.
    Compatible con Google Colab (no requiere asyncio).

    Args:
        query (str): Consulta del usuario
        candidates (pd.DataFrame): Top-30 candidatos de semantic search
        dataset_name (str): 'movies' o 'spotify'

    Returns:
        dict: {llm_name: [parsed_recommendations]}
    """
    print(f"\n🤖 Consultando {len(ACTIVE_LLMS)} LLMs {'en paralelo' if USE_PARALLEL_LLMS else 'secuencialmente'}...")

    # Construir contexto
    context = build_context(candidates, dataset_name)

    # Obtener prompt template
    prompt_template = PROMPTS[dataset_name]
    prompt = prompt_template.format(context=context, query=query)

    # OPTIMIZATION: Llamadas en paralelo con ThreadPoolExecutor
    if USE_PARALLEL_LLMS:
        from concurrent.futures import ThreadPoolExecutor, as_completed
        print(prompt)
        results = {}
        llm_names = list(ACTIVE_LLMS.keys())

        # Función helper para call individual
        def call_single_llm(llm_name):
            response = call_llm(llm_name, prompt)
            if response:
                parsed = parse_llm_json(response, llm_name)
                return llm_name, parsed, len(parsed)
            else:
                return llm_name, [], 0

        # Ejecutar en paralelo (max_workers = número de LLMs)
        with ThreadPoolExecutor(max_workers=len(llm_names)) as executor:
            # Submit all tasks
            future_to_llm = {
                executor.submit(call_single_llm, llm_name): llm_name
                for llm_name in llm_names
            }

            # Recoger resultados conforme completan
            for future in as_completed(future_to_llm):
                llm_name, parsed, count = future.result()
                results[llm_name] = parsed

                status = "✅" if count > 0 else "❌"
                msg = f"{count} recomendaciones" if count > 0 else "Sin respuesta"
                print(f"   {status} {llm_name}: {msg}")

    else:
        # Modo secuencial (original)
        results = {}
        for llm_name in ACTIVE_LLMS.keys():
            print(f"   🔄 {llm_name}...", end=" ")

            # Llamar LLM
            response = call_llm(llm_name, prompt)

            if response:
                # Parsear JSON
                parsed = parse_llm_json(response, llm_name)
                results[llm_name] = parsed
                print(f"✅ {len(parsed)} recomendaciones")
            else:
                results[llm_name] = []
                print(f"❌ Sin respuesta")

    total_recs = sum(len(recs) for recs in results.values())
    print(f"   📊 Total: {total_recs} recomendaciones de {len(results)} LLMs")

    # Mostrar resumen útil: Prompt vs Recomendaciones
    if VERBOSE:
        print("\n" + "="*70)
        print("📋 PROMPT ENVIADO A LLMs:")
        print("="*70)
        # Mostrar solo las primeras 500 caracteres del prompt
        print(prompt[:500] + "..." if len(prompt) > 500 else prompt)
        print("="*70)

    # Siempre mostrar recomendaciones recibidas (útil para debugging)
    print("\n📝 RECOMENDACIONES RECIBIDAS:")
    for llm_name, recs in results.items():
        if recs:
            print(f"\n{llm_name.upper()}:")
            for i, rec in enumerate(recs[:3], 1):  # Solo primeras 3
                print(f"  {i}. {rec['recommendation']} (score: {rec['ranking']})")
            if len(recs) > 3:
                print(f"  ... y {len(recs)-3} más")
        else:
            print(f"\n{llm_name.upper()}: ❌ Sin recomendaciones")

    return results



In [ ]:
# ═══════════════════════════════════════════════════════
# COMPONENTE 2: MULTI-SOURCE GENERATION
# ═══════════════════════════════════════════════════════


def semantic_search(query, embeddings, df, top_k=30):
    """
    Búsqueda semántica usando similitud de coseno.

    Args:
        query (str): Consulta del usuario
        embeddings (np.ndarray): Embeddings de items (n x 384)
        df (pd.DataFrame): Dataset con metadata
        top_k (int): Número de candidatos a retornar

    Returns:
        pd.DataFrame: Top-K items más similares con columna 'similarity'
    """
    # Generar embedding de query
    query_emb = encoder.encode([query])[0]

    # Calcular similitud coseno con todos los items de forma robusta
    # cosine_sim = (A · B) / (||A|| * ||B||)

    # Primero calcular normas para detectar problemas antes del matmul
    norms_embeddings = np.linalg.norm(embeddings, axis=1)
    norm_query = np.linalg.norm(query_emb)

    # Evitar división por cero (epsilon grande para estabilidad numérica)
    epsilon = 1e-8
    norms_embeddings = np.where(norms_embeddings == 0, epsilon, norms_embeddings)
    norm_query = max(norm_query, epsilon)

    # Normalizar vectores antes del dot product (evita overflow/underflow)
    embeddings_normalized = embeddings / norms_embeddings[:, np.newaxis]
    query_normalized = query_emb / norm_query

    # Dot product con vectores normalizados (más estable numéricamente)
    with np.errstate(divide='ignore', over='ignore', under='ignore', invalid='ignore'):
        similarities = embeddings_normalized @ query_normalized

    # Clip para evitar valores fuera de rango por errores numéricos
    similarities = np.clip(similarities, -1.0, 1.0)

    # Normalizar a [0, 1]: (sim + 1) / 2
    similarities_norm = (similarities + 1) / 2

    # Obtener top-K indices
    top_indices = np.argsort(similarities_norm)[-top_k:][::-1]

    # Crear DataFrame con resultados
    results = df.iloc[top_indices].copy()
    results['similarity'] = similarities_norm[top_indices]

    return results.reset_index(drop=True)


def build_context(items, dataset_name):
    """
    Construye contexto enriquecido para prompts de LLMs.

    Args:
        items (pd.DataFrame): Items candidatos (top-30)
        dataset_name (str): 'movies' o 'spotify'

    Returns:
        str: Contexto formateado para insertar en prompt
    """
    lines = []

    if dataset_name == 'movies':
        for i, row in items.iterrows():
            # Formato: [1] 296 - Pulp Fiction (1994) - Crime Drama Thriller
            # Extraer año del título si está presente
            title = row['title']
            item_id = row['id']
            genres = row['description']

            lines.append(f"{i+1}. [{item_id}] {title} - {genres}")

    elif dataset_name == 'spotify':
        for i, row in items.iterrows():
            # Formato: [1] track_id - Song by Artist - genre (dance: 0.X, energy: 0.Y)
            title = row['title']
            item_id = row['id']
            genre = row['description']

            # Agregar audio features si existen
            audio_info = ""
            if 'danceability' in row and 'energy' in row:
                audio_info = f" (dance: {row['danceability']:.2f}, energy: {row['energy']:.2f})"

            artist = row.get('artists', 'Unknown')
            lines.append(f"{i+1}. [{item_id}] {title} by {artist} - {genre}{audio_info}")
    else:
        # Fallback genérico
        for i, row in items.iterrows():
            lines.append(f"{i+1}. [{row['id']}] {row['title']} - {row['description']}")

    return '\n'.join(lines)


def call_llm(llm_name, prompt):
    """
    Llama a un LLM con retry logic y exponential backoff.

    ARQUITECTURA 100% MODEL-AGNOSTIC:
    - Usa el patrón Factory para crear proveedores
    - Cualquier modelo nuevo se agrega solo en LLMS config
    - No requiere modificar este código

    Args:
        llm_name (str): Nombre del LLM ('gpt4o', 'claude', 'gemini', etc.)
        prompt (str): Prompt completo a enviar

    Returns:
        str: Respuesta del LLM (JSON string esperado)
    """
    config = ACTIVE_LLMS[llm_name]

    for attempt in range(RETRY_ATTEMPTS):
        try:
            # Crear provider usando factory pattern (model-agnostic)
            provider_kwargs = {
                'api_key': config.get('api_key'),
                'base_url': config.get('base_url'),  # Solo para OpenAI-compatible
            }

            provider = create_llm_provider(
                provider_type=config['provider'],
                model_name=config['model'],
                **provider_kwargs
            )
            print("MAX_TOKENS " + str(MAX_TOKENS)
            # Generar respuesta (interfaz unificada)
            return provider.generate(
                prompt=prompt,
                max_tokens=MAX_TOKENS,
                temperature=TEMPERATURE
            )

        except Exception as e:
            if attempt == RETRY_ATTEMPTS - 1:
                # Último intento falló
                error_msg = f"{str(e)[:150]}"
                print(f"   ⚠️ {llm_name}: {error_msg}")

                # Mostrar traceback completo solo en modo verbose
                if VERBOSE:
                    import traceback
                    print(f"      Full error: {traceback.format_exc()[:300]}")
                return ""
            # Exponential backoff: 1s, 2s, 4s
            time.sleep(2 ** attempt)

    return ""


def parse_llm_json(response, llm_name):
    """
    Parsea respuesta JSON del LLM.

    Args:
        response (str): Respuesta del LLM (debe ser JSON array)
        llm_name (str): Nombre del LLM (para logging)

    Returns:
        list: Lista de dicts con {recommendation, ranking, description, source}
              o lista vacía si falla el parsing
    """
    if not response or not response.strip():
        return []

    try:
        # Intentar parsear JSON directamente
        data = json.loads(response)

        # Validar que sea una lista
        if not isinstance(data, list):
            print(f"   ⚠️ {llm_name}: Response no es un array JSON")
            return []

        # Validar que cada item tenga los campos requeridos
        valid_items = []
        for item in data:
            if all(k in item for k in ['recommendation', 'ranking', 'description']):
                valid_items.append({
                    'recommendation': str(item['recommendation']),
                    'ranking': float(item['ranking']),
                    'description': str(item['description']),
                    'source': str(item.get('source', 'database'))  # NUEVO: source (default: database)
                })
            else:
                print(f"   ⚠️ {llm_name}: Item sin campos requeridos: {item}")

        return valid_items

    except json.JSONDecodeError as e:
        # Intentar múltiples estrategias de limpieza

        # Estrategia 1: Extraer solo el JSON array
        try:
            start = response.find('[')
            end = response.rfind(']') + 1
            if start != -1 and end > start:
                cleaned = response[start:end]
                data = json.loads(cleaned)
                if isinstance(data, list):
                    print(f"   ℹ️ {llm_name}: JSON extraído (estrategia 1)")
                    return parse_llm_json(cleaned, llm_name)
        except:
            pass

        # Estrategia 2: Arreglar strings sin terminar
        try:
            # Buscar el error de "unterminated string"
            if "Unterminated string" in str(e):
                # Intentar cerrar strings abiertos agregando comilla antes del cierre
                cleaned = response
                # Agregar comilla de cierre si falta
                if cleaned.count('"') % 2 != 0:
                    # Hay un número impar de comillas - agregar una antes del ]
                    last_bracket = cleaned.rfind(']')
                    if last_bracket > 0:
                        cleaned = cleaned[:last_bracket] + '"' + cleaned[last_bracket:]

                data = json.loads(cleaned)
                if isinstance(data, list):
                    print(f"   ℹ️ {llm_name}: JSON reparado (comilla agregada)")
                    return parse_llm_json(cleaned, llm_name)
        except:
            pass

        # Estrategia 3: Usar regex para extraer objetos válidos
        try:
            import re
            # Buscar objetos JSON individuales
            objects = re.findall(r'\{[^}]*"recommendation"[^}]*"ranking"[^}]*"description"[^}]*\}', response)
            if objects:
                # Intentar parsear cada objeto
                valid_items = []
                for obj_str in objects:
                    try:
                        obj = json.loads(obj_str)
                        if all(k in obj for k in ['recommendation', 'ranking', 'description']):
                            valid_items.append({
                                'recommendation': str(obj['recommendation']),
                                'ranking': float(obj['ranking']),
                                'description': str(obj['description']),
                                'source': str(obj.get('source', 'database'))  # NUEVO
                            })
                    except:
                        continue

                if valid_items:
                    print(f"   ℹ️ {llm_name}: {len(valid_items)} items extraídos (regex) - esperaba 5")
                    if VERBOSE and len(valid_items) < 5:
                        print(f"      JSON raw (primeros 500 chars): {response[:500]}")
                    return valid_items
        except:
            pass

        print(f"   ⚠️ {llm_name}: JSON inválido - {str(e)[:150]}")
        if VERBOSE:
            print(f"      JSON raw (primeros 500 chars): {response[:500]}")
        return []


def query_all_llms(query, candidates, dataset_name):
    """
    Consulta todos los LLMs activos (en paralelo si USE_PARALLEL_LLMS=True).

    OPTIMIZATION: ThreadPoolExecutor permite 3-5x speedup vs secuencial.
    Compatible con Google Colab (no requiere asyncio).

    Args:
        query (str): Consulta del usuario
        candidates (pd.DataFrame): Top-30 candidatos de semantic search
        dataset_name (str): 'movies' o 'spotify'

    Returns:
        dict: {llm_name: [parsed_recommendations]}
    """
    print(f"\n🤖 Consultando {len(ACTIVE_LLMS)} LLMs {'en paralelo' if USE_PARALLEL_LLMS else 'secuencialmente'}...")

    # Construir contexto
    context = build_context(candidates, dataset_name)

    # Obtener prompt template
    prompt_template = PROMPTS[dataset_name]
    prompt = prompt_template.format(context=context, query=query)

    # OPTIMIZATION: Llamadas en paralelo con ThreadPoolExecutor
    if USE_PARALLEL_LLMS:
        from concurrent.futures import ThreadPoolExecutor, as_completed
        print(prompt)
        results = {}
        llm_names = list(ACTIVE_LLMS.keys())

        # Función helper para call individual
        def call_single_llm(llm_name):
            response = call_llm(llm_name, prompt)
            if response:
                parsed = parse_llm_json(response, llm_name)
                return llm_name, parsed, len(parsed)
            else:
                return llm_name, [], 0

        # Ejecutar en paralelo (max_workers = número de LLMs)
        with ThreadPoolExecutor(max_workers=len(llm_names)) as executor:
            # Submit all tasks
            future_to_llm = {
                executor.submit(call_single_llm, llm_name): llm_name
                for llm_name in llm_names
            }

            # Recoger resultados conforme completan
            for future in as_completed(future_to_llm):
                llm_name, parsed, count = future.result()
                results[llm_name] = parsed

                status = "✅" if count > 0 else "❌"
                msg = f"{count} recomendaciones" if count > 0 else "Sin respuesta"
                print(f"   {status} {llm_name}: {msg}")

    else:
        # Modo secuencial (original)
        results = {}
        for llm_name in ACTIVE_LLMS.keys():
            print(f"   🔄 {llm_name}...", end=" ")

            # Llamar LLM
            response = call_llm(llm_name, prompt)

            if response:
                # Parsear JSON
                parsed = parse_llm_json(response, llm_name)
                results[llm_name] = parsed
                print(f"✅ {len(parsed)} recomendaciones")
            else:
                results[llm_name] = []
                print(f"❌ Sin respuesta")

    total_recs = sum(len(recs) for recs in results.values())
    print(f"   📊 Total: {total_recs} recomendaciones de {len(results)} LLMs")

    # Mostrar resumen útil: Prompt vs Recomendaciones
    if VERBOSE:
        print("\n" + "="*70)
        print("📋 PROMPT ENVIADO A LLMs:")
        print("="*70)
        # Mostrar solo las primeras 500 caracteres del prompt
        print(prompt[:500] + "..." if len(prompt) > 500 else prompt)
        print("="*70)

    # Siempre mostrar recomendaciones recibidas (útil para debugging)
    print("\n📝 RECOMENDACIONES RECIBIDAS:")
    for llm_name, recs in results.items():
        if recs:
            print(f"\n{llm_name.upper()}:")
            for i, rec in enumerate(recs[:3], 1):  # Solo primeras 3
                print(f"  {i}. {rec['recommendation']} (score: {rec['ranking']})")
            if len(recs) > 3:
                print(f"  ... y {len(recs)-3} más")
        else:
            print(f"\n{llm_name.upper()}: ❌ Sin recomendaciones")

    return results



SyntaxError: '(' was never closed (2685058990.py, line 134)

### Componente 3: Validación Semántica

Este es el componente más crítico de la investigación. Su función es actuar como un "control de calidad" que previene las "alucinaciones" y las recomendaciones irrelevantes, incluso si provienen de fuentes confiables.

**Proceso:**

1.  **Triple Chequeo:** Cada recomendación (de LLMs y FCD) pasa por una validación triple:
    a.  **Matching de Título:** Intenta encontrar el ítem en nuestra DB (primero exacto, luego *fuzzy matching* para typos).
    b.  **Validación de Descripción (LLM):** La *descripción* dada por el LLM debe ser semánticamente similar a la consulta original del usuario.
    c.  **Validación de Ítem (DB):** El *contenido real* del ítem en nuestra DB (título + géneros) debe ser semánticamente similar a la consulta.
2.  **Métrica:** Usamos la **Similitud de Coseno** (Fórmula 1) entre los *embeddings* (vectores semánticos) de la consulta y del ítem/descripción.
3.  **Umbral:** Se aplica un umbral (`threshold`) para decidir si la similitud es suficiente. Las recomendaciones "externas" (del conocimiento del LLM, no de nuestra DB) tienen un umbral más leniente (`external_threshold`).

---

#### Fórmula 1: Similitud de Coseno
Mide la similitud entre la consulta (Q) y el ítem (R) en un espacio de 384 dimensiones.

$$ Sim(Q, R) = \\frac{Q \\cdot R}{\\\|Q\\| \\|R\\|} = \\frac{\\sum_{i=1}^{n} Q_i R_i}{\\sqrt{\\sum_{i=1}^{n} Q_i^2} \\sqrt{\\sum_{i=1}^{n} R_i^2}} $$

#### Fórmula 2: Normalización de Relevancia
Convertimos la similitud (rango [-1, 1]) a un *score* de relevancia (rango [0, 1]).

$$ relevance\\_score = \\frac{Sim(Q, R) + 1}{2} $$

---

In [ ]:
# ═══════════════════════════════════════════════════════
# COMPONENTE 3: SEMANTIC VALIDATION
# ═══════════════════════════════════════════════════════

def fuzzy_match_title(title, df, threshold=0.85):
    """
    Encuentra el título más similar en el dataset usando fuzzy string matching.
    Útil para recuperar recomendaciones con typos o variaciones del título.

    OPTIMIZATION: Usa RapidFuzz (C++ optimizado) si está disponible, 10-20x más rápido.
    Fallback a difflib si RapidFuzz no está instalado.

    Args:
        title (str): Título a buscar (puede tener errores)
        df (pd.DataFrame): Dataset donde buscar
        threshold (float): Similitud mínima requerida (0-1)

    Returns:
        tuple: (matched_title, similarity_score, item_data) o (None, 0, None)

    Ejemplo:
        fuzzy_match_title("Pulp Ficton (1994)", df, 0.85)
        # Returns: ("Pulp Fiction (1994)", 0.95, <row_data>)
    """
    # OPTIMIZATION: RapidFuzz (10-20x más rápido que difflib)
    if USE_RAPIDFUZZ:
        try:
            from rapidfuzz import process, fuzz

            # RapidFuzz busca en todas las opciones de una vez (C++ optimizado)
            result = process.extractOne(
                title,
                df['title'].tolist(),
                scorer=fuzz.ratio,
                score_cutoff=threshold * 100  # RapidFuzz usa escala 0-100
            )

            if result:
                matched_title, score_100, idx = result
                best_score = score_100 / 100  # Convertir a 0-1
                best_match = matched_title
                best_item = df.iloc[idx]
                return best_match, best_score, best_item

            return None, 0, None

        except ImportError:
            # RapidFuzz no instalado, usar fallback a difflib
            if VERBOSE:
                print("   ℹ️ RapidFuzz no disponible, usando difflib (más lento)")
            pass

    # Fallback: difflib (implementación original)
    from difflib import SequenceMatcher

    best_match = None
    best_score = 0
    best_item = None

    # Limpiar título para comparación
    title_clean = title.lower().strip()

    for idx, row in df.iterrows():
        db_title = row['title'].lower().strip()

        # Calcular similitud usando SequenceMatcher (Ratcliff/Obershelp algorithm)
        similarity = SequenceMatcher(None, title_clean, db_title).ratio()

        if similarity > best_score:
            best_score = similarity
            best_match = row['title']
            best_item = row

    if best_score >= threshold:
        return best_match, best_score, best_item

    return None, best_score, None

def semantic_validation(llm_results, query, df, dataset_name, fcd_results=None, weights=None):
    """
    Validación semántica MEJORADA con triple check y soporte para recomendaciones externas + FCD:
    1. Title matching (exact → fuzzy → external)
    2. Description del LLM es semánticamente similar al query
    3. Item content es semánticamente similar al query (solo para items del dataset)
    4. FCD recommendations: validación directa por item_id

    NUEVO: Soporta recomendaciones del conocimiento interno del LLM + FCD

    Args:
        llm_results (dict): {llm_name: [recommendations]}
        query (str): Query original del usuario
        df (pd.DataFrame): Dataset completo
        dataset_name (str): 'movies' o 'spotify'
        fcd_results (dict): {'fcd': [recommendations]} (opcional)
        weights (dict): Pesos de balanceo (opcional)

    Returns:
        list: Items validados con metadata completa (incluye flag is_external y source)
    """
    if fcd_results is None:
        fcd_results = {}
    if weights is None:
        weights = {'llm': 1.0, 'fcd': 0.0}
    config = DATASETS[dataset_name]
    threshold = config['threshold']
    external_threshold = config.get('external_threshold', threshold * 0.85)
    allow_external = config.get('allow_external', False)

    print(f"\n🔍 Validación Semántica Mejorada:")
    print(f"   Database threshold: {threshold}")
    print(f"   External threshold: {external_threshold}")
    print(f"   Allow external: {allow_external}")

    query_emb = encoder.encode([query])[0]

    validated_items = []
    stats = {
        'total': 0,
        'exact_match': 0,
        'fuzzy_match': 0,
        'external_valid': 0,
        'external_invalid': 0,
        'desc_invalid': 0,
        'item_invalid': 0,
    }

    for llm_name, recommendations in llm_results.items():
        for rec in recommendations:
            stats['total'] += 1

            title = rec['recommendation']
            source = rec.get('source', 'database')  # Default: database

            # ═══════════════════════════════════════════════════════
            # STEP 1: TITLE MATCHING (3 niveles: exact → fuzzy → external)
            # ═══════════════════════════════════════════════════════

            # Nivel 1a: Exact match
            matching_items = df[df['title'].str.contains(title, case=False, regex=False, na=False)]

            if len(matching_items) > 0:
                # ✅ EXACT MATCH encontrado
                stats['exact_match'] += 1
                item_data = matching_items.iloc[0]
                match_type = 'exact'
                match_score = 1.0
                is_external = False

            else:
                # Nivel 1b: Fuzzy matching (typos/variaciones)
                fuzzy_title, fuzzy_score, fuzzy_item = fuzzy_match_title(title, df, threshold=0.85)

                if fuzzy_title:
                    # ✅ FUZZY MATCH encontrado
                    stats['fuzzy_match'] += 1
                    item_data = fuzzy_item
                    match_type = 'fuzzy'
                    match_score = fuzzy_score
                    is_external = False
                    if VERBOSE:
                        print(f"   ℹ️ {llm_name}: Fuzzy match '{title}' → '{fuzzy_title}' (score={fuzzy_score:.3f})")

                elif source == 'external' and allow_external:
                    # Nivel 1c: External recommendation (del conocimiento del LLM)
                    if VERBOSE:
                        print(f"   ℹ️ {llm_name}: External recommendation '{title}' (not in database)")

                    # Para externos, solo validamos la descripción (no podemos validar el item)
                    desc_emb = encoder.encode([rec['description']])[0]
                    desc_similarity = np.dot(query_emb, desc_emb) / (
                        np.linalg.norm(query_emb) * np.linalg.norm(desc_emb) + 1e-10
                    )
                    desc_similarity = np.clip(desc_similarity, -1.0, 1.0)
                    desc_similarity_norm = (desc_similarity + 1) / 2

                    if desc_similarity_norm >= external_threshold:
                        # ✅ EXTERNAL VÁLIDO (descripción suficientemente relevante)
                        stats['external_valid'] += 1
                        validated_items.append({
                            'llm_source': llm_name,
                            'title': title,
                            'item_id': f"EXTERNAL_{len(validated_items)}",
                            'description': 'External recommendation',
                            'llm_score': rec['ranking'],
                            'llm_reasoning': rec['description'],
                            'desc_similarity': desc_similarity_norm,
                            'item_similarity': desc_similarity_norm * 0.9,  # Penalty 10%
                            'relevance_score': desc_similarity_norm * 0.85,  # Penalty 15%
                            'match_type': 'external',
                            'match_score': 0.0,
                            'is_external': True,
                        })
                        continue
                    else:
                        # ❌ EXTERNAL RECHAZADO (descripción no suficientemente relevante)
                        stats['external_invalid'] += 1
                        if VERBOSE:
                            print(f"   ⚠️ {llm_name}: External '{title}' rejected (desc_sim={desc_similarity_norm:.3f} < {external_threshold})")
                        continue

                else:
                    # ❌ NO ENCONTRADO y no es external permitido
                    stats['external_invalid'] += 1
                    if VERBOSE:
                        print(f"   ⚠️ {llm_name}: '{title}' not found (no exact/fuzzy match)")
                    continue

            # ═══════════════════════════════════════════════════════
            # STEP 2: DESCRIPTION VALIDATION (para items del dataset)
            # ═══════════════════════════════════════════════════════

            desc_emb = encoder.encode([rec['description']])[0]
            desc_similarity = np.dot(query_emb, desc_emb) / (
                np.linalg.norm(query_emb) * np.linalg.norm(desc_emb) + 1e-10
            )
            desc_similarity = np.clip(desc_similarity, -1.0, 1.0)
            desc_similarity_norm = (desc_similarity + 1) / 2

            if desc_similarity_norm < threshold:
                stats['desc_invalid'] += 1
                if VERBOSE:
                    print(f"   ⚠️ {llm_name}: Description of '{title}' not similar ({desc_similarity_norm:.3f} < {threshold})")
                continue

            # ═══════════════════════════════════════════════════════
            # STEP 3: ITEM VALIDATION (para items del dataset)
            # ═══════════════════════════════════════════════════════

            item_text = f"{item_data['title']}. {item_data['description']}"
            item_emb = encoder.encode([item_text])[0]
            item_similarity = np.dot(query_emb, item_emb) / (
                np.linalg.norm(query_emb) * np.linalg.norm(item_emb) + 1e-10
            )
            item_similarity = np.clip(item_similarity, -1.0, 1.0)
            item_similarity_norm = (item_similarity + 1) / 2

            if item_similarity_norm < threshold:
                stats['item_invalid'] += 1
                if VERBOSE:
                    print(f"   ⚠️ {llm_name}: Item '{title}' not similar ({item_similarity_norm:.3f} < {threshold})")
                continue

            # ✅ ITEM VALIDADO (del dataset)
            validated_items.append({
                'llm_source': llm_name,
                'title': item_data['title'],
                'item_id': item_data['id'],
                'description': item_data['description'],
                'llm_score': rec['ranking'],
                'llm_reasoning': rec['description'],
                'desc_similarity': desc_similarity_norm,
                'item_similarity': item_similarity_norm,
                'relevance_score': (desc_similarity_norm + item_similarity_norm) / 2,
                'match_type': match_type,
                'match_score': match_score,
                'is_external': is_external,
            })

    # ═══════════════════════════════════════════════════════
    # VALIDACIÓN DE FCD RECOMMENDATIONS
    # ═══════════════════════════════════════════════════════

    fcd_stats = {'total': 0, 'valid': 0, 'invalid': 0}

    if fcd_results:
        print(f"\n🤝 Validando recomendaciones de FCD...")

        query_emb = encoder.encode([query])[0]
        id_col = 'id'

        for source_name, recommendations in fcd_results.items():
            for fcd_rec in recommendations:
                fcd_stats['total'] += 1

                item_id = fcd_rec['item_id']

                # Buscar item en el dataset
                matching_items = df[df[id_col].astype(str) == str(item_id)]

                if len(matching_items) == 0:
                    fcd_stats['invalid'] += 1
                    if VERBOSE:
                        print(f"   ⚠️ FCD: Item {item_id} not found in dataset")
                    continue

                item_data = matching_items.iloc[0]

                # Validar similitud semántica del item con la query
                item_text = f"{item_data['title']}. {item_data['description']}"
                item_emb = encoder.encode([item_text])[0]
                item_similarity = np.dot(query_emb, item_emb) / (
                    np.linalg.norm(query_emb) * np.linalg.norm(item_emb) + 1e-10
                )
                item_similarity = np.clip(item_similarity, -1.0, 1.0)
                item_similarity_norm = (item_similarity + 1) / 2

                # Mismo umbral que LLMs (validación estricta)
                # FCD solo debe hacer boost a items YA relevantes, no bypass validación
                fcd_threshold = threshold  # 0.60 para movies (no más leniente)

                if item_similarity_norm < fcd_threshold:
                    fcd_stats['invalid'] += 1
                    if VERBOSE:
                        print(f"   ⚠️ FCD: Item '{item_data['title']}' not similar ({item_similarity_norm:.3f} < {fcd_threshold:.3f})")
                    continue

                # ✅ FCD ITEM VALIDADO
                fcd_stats['valid'] += 1

                validated_items.append({
                    'llm_source': 'fcd',
                    'title': item_data['title'],
                    'item_id': item_data['id'],
                    'description': item_data['description'],
                    'llm_score': 0.0,  # FCD no usa este score
                    'llm_reasoning': f"Recommended by collaborative filtering (score: {fcd_rec['fcd_score']:.3f})",
                    'desc_similarity': item_similarity_norm,
                    'item_similarity': item_similarity_norm,
                    'relevance_score': item_similarity_norm,
                    'match_type': 'fcd',
                    'match_score': 1.0,
                    'is_external': False,
                    'fcd_score': fcd_rec['fcd_score'],  # Score original de FCD
                    'fcd_n_recs': fcd_rec.get('n_recommendations', 0),
                })

        if fcd_stats['total'] > 0:
            print(f"   📊 FCD validados: {fcd_stats['valid']}/{fcd_stats['total']}")

    # ═══════════════════════════════════════════════════════
    # ESTADÍSTICAS MEJORADAS
    # ═══════════════════════════════════════════════════════

    total_valid = len(validated_items)
    print(f"\n   📊 Validación Completa:")
    print(f"      Total LLMs procesados: {stats['total']}")
    print(f"      ✅ Exact matches: {stats['exact_match']}")
    print(f"      ✅ Fuzzy matches: {stats['fuzzy_match']}")
    if allow_external:
        print(f"      ✅ External (valid): {stats['external_valid']}")
        print(f"      ❌ External (invalid): {stats['external_invalid']}")
    print(f"      ❌ Description invalid: {stats['desc_invalid']}")
    print(f"      ❌ Item invalid: {stats['item_invalid']}")
    if fcd_stats['total'] > 0:
        print(f"      Total FCD procesados: {fcd_stats['total']}")
        print(f"      ✅ FCD valid: {fcd_stats['valid']}")
        print(f"      ❌ FCD invalid: {fcd_stats['invalid']}")
    print(f"      ✅ TOTAL VALIDATED: {total_valid}")

    return validated_items



In [ ]:
# ═══════════════════════════════════════════════════════
# OPTIMIZACIONES NIVEL 1: RAG DIAGNOSTICS
# ═══════════════════════════════════════════════════════

def measure_rag_recall(query, ground_truth, collection, top_k=200):
    """
    Mide el Recall@K del RAG: ¿cuántos items relevantes están en top-K?

    Esta métrica es crítica para diagnosticar si el RAG es el cuello de botella.
    Si Recall@200 < 0.75, significa que muchos items relevantes no son recuperados,
    y necesitamos Query Expansion.

    Args:
        query (str): Query del usuario
        ground_truth (set): Items relevantes (títulos del dataset)
        collection: ChromaDB collection
        top_k (int): Candidatos recuperados (default: 200)

    Returns:
        dict: {
            'recall': float (0-1),
            'hits': int,
            'total_relevant': int,
            'missing_items': set
        }

    Ejemplo:
        >>> ground_truth = {"Primer (2004)", "Inception (2010)", "Interstellar (2014)"}
        >>> result = measure_rag_recall(query, ground_truth, collection, top_k=200)
        >>> print(f"Recall@200: {result['recall']:.2f}")
        Recall@200: 0.67  # Solo 2 de 3 encontrados
    """
    candidates = rag_retrieval(query, collection, top_k=top_k)
    retrieved_titles = set(candidates['title'].tolist())

    hits = len(ground_truth & retrieved_titles)
    total_relevant = len(ground_truth)
    recall = hits / total_relevant if total_relevant > 0 else 0.0

    missing_items = ground_truth - retrieved_titles

    print(f"\n📊 RAG Recall@{top_k} Diagnostic:")
    print(f"   Recall: {recall:.2%} ({hits}/{total_relevant} relevantes recuperados)")
    if missing_items:
        print(f"   Missing items ({len(missing_items)}):")
        for item in list(missing_items)[:5]:
            print(f"      • {item}")
        if len(missing_items) > 5:
            print(f"      ... y {len(missing_items) - 5} más")

    if recall < 0.75:
        print(f"   ⚠️  ADVERTENCIA: Recall bajo ({recall:.2%}). Considera Query Expansion.")
    else:
        print(f"   ✅ Recall aceptable ({recall:.2%}). RAG no es el cuello de botella.")

    return {
        'recall': recall,
        'hits': hits,
        'total_relevant': total_relevant,
        'missing_items': missing_items
    }


# ═══════════════════════════════════════════════════════
# OPTIMIZACIONES NIVEL 1: QUERY EXPANSION
# ═══════════════════════════════════════════════════════

def expand_query_with_llm(query, expansion_count=4):
    """
    Expande query en múltiples variantes usando LLM.

    Beneficio: Consultas complejas ("sci-fi time travel mind-bending") son difíciles
    de capturar con un solo embedding. Query expansion genera variantes más simples
    que mejoran el recall del RAG.

    Args:
        query (str): Query original del usuario
        expansion_count (int): Número de variantes a generar (default: 4)

    Returns:
        list[str]: [query_original, variante1, variante2, ...]

    Ejemplo:
        >>> expand_query_with_llm("sci-fi time travel mind-bending movies")
        [
            "sci-fi time travel mind-bending movies",
            "time travel movies with plot twists",
            "mind-bending science fiction films",
            "complex timeline movies",
            "Christopher Nolan style sci-fi"
        ]
    """
    # Usar LLM rápido y barato para expansión
    # Prioridad: Gemini (gratis) > GPT-4o-mini > Claude Haiku
    llm_config = None
    if 'gemini' in ACTIVE_LLMS:
        llm_config = ACTIVE_LLMS['gemini']
    elif 'gpt4o' in ACTIVE_LLMS:
        llm_config = ACTIVE_LLMS['gpt4o']
    elif 'claude' in ACTIVE_LLMS:
        llm_config = ACTIVE_LLMS['claude']
    else:
        # Sin LLM disponible → retornar solo query original
        if VERBOSE:
            print("   ⚠️ No hay LLM disponible para query expansion")
        return [query]

    try:
        provider = create_llm_provider(
            llm_config['provider'],
            llm_config['model'],
            api_key=llm_config['api_key'],
            base_url=llm_config.get('base_url')
        )

        prompt = f"""Given this movie search query: "{query}"

Generate {expansion_count} alternative phrasings that capture different aspects of the query.
Each alternative should be shorter and more specific, focusing on different keywords.

Return ONLY a JSON array of strings, no additional text.

Example:
["alternative 1", "alternative 2", "alternative 3", "alternative 4"]

Response:"""

        response = provider.generate(prompt, max_tokens=150, temperature=0.7)

        # Parse JSON
        try:
            alternatives = json.loads(response.strip())
            if isinstance(alternatives, list) and len(alternatives) > 0:
                return [query] + alternatives[:expansion_count]
        except json.JSONDecodeError:
            if VERBOSE:
                print(f"   ⚠️ Error parsing query expansion JSON: {response[:100]}")

    except Exception as e:
        if VERBOSE:
            print(f"   ⚠️ Error en query expansion: {e}")

    # Fallback: retornar solo query original
    return [query]


def rag_retrieval_expanded(query, collection, top_k=30, expansion_count=4, use_expansion=True):
    """
    RAG con query expansion para mejorar recall en consultas complejas.

    Pipeline:
    1. Expandir query en N variantes (usando LLM)
    2. Ejecutar RAG retrieval para cada variante (batch)
    3. Fusionar y deduplicar resultados
    4. Re-rankear por max similarity

    Args:
        query (str): Query original del usuario
        collection: ChromaDB collection
        top_k (int): Candidatos a retornar (default: 30)
        expansion_count (int): Número de expansiones (default: 4)
        use_expansion (bool): Si False, usa RAG normal (para comparación)

    Returns:
        pd.DataFrame: Top-K candidatos fusionados

    Benchmark esperado:
        - Recall@200: +15-25%
        - P@10: +10-15%
        - Latencia: +3-5s (5 queries vs 1)
    """
    if not use_expansion:
        # Fallback: RAG normal
        return rag_retrieval(query, collection, top_k=top_k)

    print(f"\n🔍 Query Expansion (generating {expansion_count} variants)...")

    # Expandir query
    expanded_queries = expand_query_with_llm(query, expansion_count=expansion_count)

    if VERBOSE:
        print(f"   Expanded queries ({len(expanded_queries)}):")
        for i, q in enumerate(expanded_queries):
            print(f"      {i+1}. {q}")

    # Recuperar para cada query expandida (batch processing)
    all_results = rag_retrieval_batch(expanded_queries, collection, top_k=top_k)

    # Fusionar y deduplicar
    merged = pd.concat(all_results).drop_duplicates(subset='id')

    # Re-rankear por max similarity (un item puede aparecer en múltiples queries)
    merged = merged.groupby('id').agg({
        'title': 'first',
        'description': 'first',
        'similarity': 'max'  # Tomar máxima similitud
    }).reset_index()

    # Ordenar por similarity
    merged = merged.sort_values('similarity', ascending=False).head(top_k * 2)

    print(f"   ✅ {len(merged)} candidatos únicos (fusionados de {len(expanded_queries)} queries)")
    print(f"   📊 Similarity: avg={merged['similarity'].mean():.3f}, max={merged['similarity'].max():.3f}")

    return merged


# ═══════════════════════════════════════════════════════
# OPTIMIZACIONES NIVEL 1: RECIPROCAL RANK FUSION (RRF)
# ═══════════════════════════════════════════════════════

def reciprocal_rank_fusion(ranked_lists, k=60, weights=None):
    """
    Fusiona múltiples listas rankeadas usando Reciprocal Rank Fusion (RRF).

    RRF es un método SOTA para agregar rankings de múltiples fuentes.
    Es robusto a diferencias en escalas de scores y se enfoca en la
    preferencia de orden, mejorando directamente nDCG.

    Fórmula:
        RRF_Score(d) = Σ weight_s / (k + rank_s(d))
                       s ∈ Sources

    Args:
        ranked_lists (dict): {source_name: [(item_id, item_data), ...]}
            - source_name: 'rag', 'fcd', 'gpt4o', 'claude', etc.
            - item_data: dict con metadata del item
        k (int): Constante de RRF (default: 60, valor estándar)
        weights (dict): Pesos por fuente (default: igual para todos)

    Returns:
        list: Items ordenados por RRF_Score descendente

    Ejemplo:
        >>> ranked_lists = {
        ...     'rag': [('item1', data1), ('item2', data2), ...],
        ...     'gpt4o': [('item3', data3), ('item1', data1), ...],
        ...     'claude': [('item1', data1), ('item4', data4), ...]
        ... }
        >>> results = reciprocal_rank_fusion(ranked_lists, k=60)
        >>> # 'item1' aparece en top-3 de 3 fuentes → RRF alto

    Referencias:
        Cormack et al. (2009). Reciprocal rank fusion outperforms
        condorcet and individual rank learning methods.
    """
    if weights is None:
        # Pesos iguales para todas las fuentes
        weights = {source: 1.0 for source in ranked_lists.keys()}

    rrf_scores = {}
    item_metadata = {}  # Guardar metadata del primer item encontrado

    # Calcular RRF score para cada item
    for source_name, ranked_items in ranked_lists.items():
        source_weight = weights.get(source_name, 1.0)

        for rank, (item_id, item_data) in enumerate(ranked_items, start=1):
            # Inicializar si es primera vez que vemos este item
            if item_id not in rrf_scores:
                rrf_scores[item_id] = 0.0
                item_metadata[item_id] = item_data

            # Agregar contribución de esta fuente
            rrf_scores[item_id] += source_weight / (k + rank)

    # Ordenar por RRF score descendente
    sorted_items = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

    # Agregar metadata
    results = []
    for item_id, rrf_score in sorted_items:
        item = item_metadata[item_id].copy()
        item['rrf_score'] = rrf_score
        item['item_id'] = item_id
        results.append(item)

    return results


def calculate_ranking_rrf(validated_items, weights=None):
    """
    Calcula ranking usando RRF (reemplaza fórmula lineal).

    Pipeline:
    1. Agrupar items por fuente (LLMs, FCD, RAG)
    2. Para cada fuente, crear lista rankeada
    3. Aplicar RRF para fusionar todas las fuentes
    4. Retornar items ordenados por RRF_Score

    Args:
        validated_items (list): Items validados con metadata
        weights (dict): Pesos por tipo de fuente {'llm': 1.0, 'fcd': 0.5}

    Returns:
        list: Items rankeados por RRF

    Beneficios vs fórmula lineal:
        - nDCG: +20-30% (mejor orden)
        - Robusto a escalas de scores diferentes
        - Aprovecha "sabiduría colectiva" de múltiples fuentes
    """
    print(f"\n📊 Ranking con RRF ({len(validated_items)} items)...")

    if not validated_items:
        print("   ⚠️ No hay items para rankear")
        return []

    # Agrupar items por fuente
    sources = {}

    for item in validated_items:
        source_name = item['llm_source']

        if source_name not in sources:
            sources[source_name] = []

        sources[source_name].append(item)

    # Ordenar cada fuente por su score específico
    ranked_lists = {}

    for source_name, items in sources.items():
        # Ordenar por relevance_score (común a todas las fuentes)
        sorted_items = sorted(items, key=lambda x: x['relevance_score'], reverse=True)

        # Convertir a formato RRF: [(item_id, item_data), ...]
        ranked_lists[source_name] = [
            (item['title'], item) for item in sorted_items
        ]

    # Aplicar RRF
    if weights is None:
        weights = {'llm': 1.0, 'fcd': 1.0}

    # Ajustar pesos: LLMs tienen peso 'llm', FCD tiene peso 'fcd'
    rrf_weights = {}
    for source_name in ranked_lists.keys():
        if source_name == 'fcd':
            rrf_weights[source_name] = weights.get('fcd', 1.0)
        else:
            rrf_weights[source_name] = weights.get('llm', 1.0)

    results = reciprocal_rank_fusion(ranked_lists, k=60, weights=rrf_weights)

    # Agregar campos de relevance, consensus y mention_count
    for item in results:
        # Encontrar todas las menciones de este item en las fuentes
        item_mentions = []
        for source_name, ranked_list in ranked_lists.items():
            for title, item_data in ranked_list:
                if title == item['item_id']:
                    item_mentions.append(item_data)
                    break

        # Calcular métricas
        item['mention_count'] = len(item_mentions)
        item['consensus'] = len(item_mentions) / len(ranked_lists) if len(ranked_lists) > 0 else 0

        # Relevance: promedio de relevance_score de todas las menciones
        if item_mentions:
            item['relevance'] = sum(m.get('relevance_score', 0.5) for m in item_mentions) / len(item_mentions)
        else:
            item['relevance'] = item.get('relevance_score', 0.5)

        # rank_score = rrf_score (ya calculado)
        item['rank_score'] = item['rrf_score']

    print(f"   ✅ {len(results)} items únicos rankeados con RRF")
    print(f"      Top-3:")
    for i, item in enumerate(results[:3], 1):
        print(f"         {i}. {item['title'][:50]:50s} (rrf={item['rrf_score']:.4f}, sources={item['mention_count']})")

    return results


In [ ]:

# ═══════════════════════════════════════════════════════
# OPTIMIZACIONES NIVEL 2: LLM RE-RANKER
# ═══════════════════════════════════════════════════════

def llm_reranker(query, candidate_items, dataset_name, top_k=10, use_reranker=True):
    """
    Re-rankea candidatos usando un LLM potente (GPT-4o, Claude Opus).

    Esta es una de las optimizaciones más potentes: usa un LLM de alta calidad
    NO para generar recomendaciones, sino para ORDENARLAS.

    Pipeline:
    1. Recibir ~30 candidatos validados
    2. Enviar al LLM con instrucciones de re-ranking
    3. LLM retorna índices ordenados [15, 3, 7, 22, ...]
    4. Retornar items en orden del LLM

    Args:
        query (str): Query original del usuario
        candidate_items (list): Items candidatos (30-50)
        dataset_name (str): 'movies' o 'spotify'
        top_k (int): Número de items a retornar (default: 10)
        use_reranker (bool): Si False, usa orden actual (para comparación)

    Returns:
        list: Top-K items ordenados por LLM

    Impacto esperado:
        - nDCG: +30-50% (orden óptimo)
        - P@10: +15-35% (items más relevantes arriba)
        - Latencia: +2-4s (1 llamada LLM adicional)

    Referencias:
        Sun et al. (2023). Is ChatGPT Good at Search?
        Investigating Large Language Models as Re-Ranking Agent.
    """
    if not use_reranker:
        # Fallback: retornar top-K sin re-ranking
        return candidate_items[:top_k]

    print(f"\n🔧 Re-Ranking con LLM ({len(candidate_items)} candidates → top-{min(top_k, len(candidate_items))})...")

    # Seleccionar LLM más potente disponible
    # Prioridad: GPT-4o > Claude > Gemini
    llm_config = None
    llm_name = None

    if 'gpt4o' in ACTIVE_LLMS:
        llm_config = ACTIVE_LLMS['gpt4o']
        llm_name = 'gpt4o'
    elif 'claude' in ACTIVE_LLMS:
        llm_config = ACTIVE_LLMS['claude']
        llm_name = 'claude'
    elif 'gemini' in ACTIVE_LLMS:
        llm_config = ACTIVE_LLMS['gemini']
        llm_name = 'gemini'
    else:
        print("   ⚠️ No hay LLM disponible para re-ranking, usando orden actual")
        return candidate_items[:top_k]

    try:
        provider = create_llm_provider(
            llm_config['provider'],
            llm_config['model'],
            api_key=llm_config['api_key'],
            base_url=llm_config.get('base_url')
        )

        # Construir prompt con candidatos numerados
        candidates_text = "\n".join([
            f"{i+1}. {item['title']} - {item.get('description', 'No description')[:100]}"
            for i, item in enumerate(candidate_items)
        ])

        domain = "movies" if dataset_name == 'movies' else "music tracks"

        prompt = f"""User query: "{query}"

Re-rank the following candidate {domain} based on how well they match the user's query.
Consider relevance, thematic alignment, and quality.

Candidates:
{candidates_text}

Instructions:
1. Analyze each candidate's relevance to the query
2. Return the top {top_k} candidates, ordered from most to least relevant
3. Return ONLY a JSON array of numbers (candidate indices 1-{len(candidate_items)}), no explanation

Example output:
[15, 3, 7, 22, 1, 9, 18, 4, 12, 6]

Response:"""

        response = provider.generate(prompt, max_tokens=100, temperature=0.3)

        # Parse indices
        try:
            # Limpiar response (a veces LLMs agregan texto extra)
            response_clean = response.strip()
            if '```' in response_clean:
                # Extraer JSON de markdown code block
                response_clean = response_clean.split('```')[1]
                if response_clean.startswith('json'):
                    response_clean = response_clean[4:]

            indices = json.loads(response_clean.strip())

            # Validar indices
            if isinstance(indices, list) and len(indices) > 0:
                # Convertir a 0-indexed y filtrar inválidos
                valid_indices = [
                    idx - 1 for idx in indices
                    if isinstance(idx, int) and 0 < idx <= len(candidate_items)
                ][:top_k]

                if len(valid_indices) >= top_k // 2:
                    # Si al menos la mitad de indices son válidos, usar re-ranking
                    reranked = [candidate_items[idx] for idx in valid_indices]

                    # Completar con items restantes si no hay suficientes
                    if len(reranked) < top_k:
                        remaining = [
                            item for i, item in enumerate(candidate_items)
                            if i not in valid_indices
                        ]
                        reranked.extend(remaining[:top_k - len(reranked)])

                    print(f"   ✅ LLM Re-Ranker ({llm_name}): {len(reranked)} items ordenados")
                    if VERBOSE:
                        for i, item in enumerate(reranked[:3], 1):
                            print(f"      {i}. {item['title'][:50]}")

                    return reranked

        except json.JSONDecodeError as e:
            if VERBOSE:
                print(f"   ⚠️ Error parsing re-ranker JSON: {e}")
                print(f"      Response: {response[:200]}")

    except Exception as e:
        print(f"   ⚠️ Error en LLM Re-Ranker: {e}")

    # Fallback: retornar orden original
    print(f"   ℹ️ Usando orden original (re-ranker falló)")
    return candidate_items[:top_k]


# ═══════════════════════════════════════════════════════
# OPTIMIZACIONES NIVEL 2: ARQUITECTURA HÍBRIDA 2 ETAPAS
# ═══════════════════════════════════════════════════════

def two_stage_hybrid_ranking(validated_items, user_id, dataset_name, top_k=10, use_two_stage=True):
    """
    Arquitectura híbrida de dos etapas: Relevance Filtering → Personalization.

    ETAPA 1: Relevance Filtering (RRF sin FCD)
        - Fusiona rankings de todas las fuentes LLM
        - Resultado: Top-50 items más RELEVANTES (sin personalización)

    ETAPA 2: Personalization Re-Ranking (FCD)
        - Para estos 50 items, obtener FCD scores
        - Combinar: final_score = 0.5 * rrf_score + 0.5 * fcd_score
        - Resultado: Top-K PERSONALIZADOS

    Args:
        validated_items (list): Items validados por LLMs
        user_id (int): ID del usuario (None = sin personalización)
        dataset_name (str): 'movies' o 'spotify'
        top_k (int): Número de recomendaciones finales
        use_two_stage (bool): Si False, usa ranking simple (para comparación)

    Returns:
        list: Top-K items personalizados

    Beneficios:
        - Items SOLO de FCD no son penalizados
        - Separación clara: relevancia (universal) vs personalización (individual)
        - FCD solo se ejecuta para 50 items (más eficiente)
        - P@10: +5-10%, nDCG: +5-10%
    """
    if not use_two_stage or user_id is None:
        # Fallback: ranking simple con RRF
        ranked = calculate_ranking_rrf(validated_items, weights=None)
        return ranked[:top_k]

    print(f"\n🎯 Arquitectura Híbrida de 2 Etapas (user_id={user_id})...")

    # ETAPA 1: Relevance Filtering (RRF sin FCD)
    print("\n   Etapa 1: Relevance Filtering (RRF, sin FCD)...")

    # Filtrar solo items de LLMs (excluir FCD temporalmente)
    llm_items = [item for item in validated_items if item['llm_source'] != 'fcd']

    if not llm_items:
        print("   ⚠️ No hay items de LLMs, usando FCD solo")
        return validated_items[:top_k]

    # Aplicar RRF solo a LLMs
    ranked_relevant = calculate_ranking_rrf(llm_items, weights={'llm': 1.0})

    # Top-50 más relevantes
    top_50_relevant = ranked_relevant[:min(50, len(ranked_relevant))]
    print(f"      ✅ Top-50 items más relevantes (RRF)")

    # ETAPA 2: Personalization Re-Ranking (FCD)
    print("\n   Etapa 2: Personalization Re-Ranking (RRF + FCD)...")

    try:
        # Cargar datos de FCD
        from recsys_multi_llm import load_user_ratings, fcd_recommendations, load_data

        ratings_df, user_item_matrix, user_id_map, item_id_map = load_user_ratings(dataset_name)

        # Obtener FCD scores para estos 50 items específicos
        fcd_recs_all = fcd_recommendations(
            user_id, dataset_name, ratings_df,
            user_item_matrix, user_id_map, item_id_map, top_k=100
        )

        # Cargar dataset para mapear item_id → title
        df, _ = load_data(dataset_name)

        # Mapear FCD scores por título
        fcd_scores_map = {}
        for rec in fcd_recs_all:
            item_id = rec['item_id']
            # Buscar título en el dataset
            matching_items = df[df['id'].astype(str) == str(item_id)]
            if len(matching_items) > 0:
                title = matching_items.iloc[0]['title']
                fcd_scores_map[title] = rec['fcd_score']

        # Combinar RRF + FCD
        personalized_items = []

        for item in top_50_relevant:
            # Obtener título con fallback a item_id
            title = item.get('title', item.get('item_id', ''))

            if not title:
                if VERBOSE:
                    print(f"   ⚠️ Item sin título: {item}")
                continue

            # RRF score (normalizar a [0, 1])
            rrf_score = item.get('rrf_score', 0.0)

            # FCD score (ya está en [0, 1])
            fcd_score = fcd_scores_map.get(title, 0.0)

            # Combinar 50/50
            final_score = 0.5 * rrf_score + 0.5 * fcd_score

            item_combined = item.copy()
            item_combined['title'] = title  # Asegurar que title existe
            item_combined['final_score'] = final_score
            item_combined['rrf_score_normalized'] = rrf_score
            item_combined['fcd_score'] = fcd_score

            personalized_items.append(item_combined)

        # Ordenar por final_score
        personalized_items.sort(key=lambda x: x['final_score'], reverse=True)

        print(f"      ✅ Top-{top_k} personalizados (RRF + FCD)")
        if VERBOSE:
            for i, item in enumerate(personalized_items[:3], 1):
                print(f"         {i}. {item['title'][:50]:50s} "
                      f"(final={item['final_score']:.3f}, "
                      f"rrf={item['rrf_score_normalized']:.3f}, "
                      f"fcd={item['fcd_score']:.3f})")

        return personalized_items[:top_k]

    except Exception as e:
        print(f"   ⚠️ Error en Etapa 2 (FCD): {e}")
        if VERBOSE:
            import traceback
            traceback.print_exc()

        # Fallback: retornar top-K de etapa 1
        print("   ℹ️ Usando solo Etapa 1 (sin personalización FCD)")
        return top_50_relevant[:top_k]



### Componente 4: Ranking

Una vez que tenemos una lista de ítems *validados*, los rankeamos. El objetivo es que los mejores ítems (alta relevancia y alto consenso) suban a la cima.

Primero, deduplicamos la lista (ej. "Interstellar" pudo ser recomendada por GPT-4, Claude y FCD). Al agrupar, calculamos dos métricas clave:

1.  **Relevancia (Relevance):** El `relevance_score` más alto (Fórmula 2) de todas las menciones.
2.  **Consenso (Consensus):** Cuántos LLMs (no FCD) mencionaron el ítem, normalizado por el total de LLMs. (ej. 3 de 5 LLMs = 0.6).
3.  **Score FCD:** El `fcd_score` (si existe).

---

#### Fórmula 3: Puntuación de Ranking Adaptativo

Usamos una fórmula de agregación ponderada que se adapta dinámicamente si el FCD está activo (basado en los `weights` del Componente 1).

$ \\alpha = 0.6 \\times weight_{llm} $ (Peso de la Relevancia)
$ \\beta = 0.4 \\times weight_{llm} $ (Peso del Consenso)
$ \\gamma = weight_{fcd} $ (Peso del FCD)

$$ rank\\_score = (\\alpha \\times relevance) + (\\beta \\times consensus) + (\\gamma \\times fcd\\_score) $$

Si un usuario es *nuevo* (`weight_{llm}=1.0`, `weight_{fcd}=0.0`), la fórmula se simplifica a:

$$ rank\\_score = (0.6 \\times relevance) + (0.4 \\times consensus) $$

In [ ]:
# ═══════════════════════════════════════════════════════
# COMPONENTE 4: RANKING (ORIGINAL - DEPRECATED)
# ═══════════════════════════════════════════════════════
# NOTA: Esta función será reemplazada por calculate_ranking_rrf en el pipeline
#       Se mantiene para compatibilidad con código legacy.

def calculate_ranking(validated_items, weights=None):
    """
    Calcula rank_score combinando relevancia, consenso y FCD (si aplica).

    Fórmula adaptativa:
    - Sin FCD: rank_score = 0.6 * relevance + 0.4 * consensus
    - Con FCD: rank_score = α * relevance + β * consensus + γ * fcd_score
      donde α = 0.6 * weight_llm, β = 0.4 * weight_llm, γ = weight_fcd

    Args:
        validated_items (list): Items validados con metadata
        weights (dict): Pesos de balanceo {'llm': 0.7, 'fcd': 0.3}

    Returns:
        list: Items rankeados y ordenados por rank_score
    """
    if weights is None:
        weights = {'llm': 1.0, 'fcd': 0.0}

    print(f"\n📊 Ranking ({len(validated_items)} items)...")

    if not validated_items:
        print("   ⚠️ No hay items para rankear")
        return []

    # Determinar si hay items de FCD
    has_fcd = any('fcd_score' in item for item in validated_items)

    # Calcular coeficientes de la fórmula
    if has_fcd and weights['fcd'] > 0:
        alpha = 0.6 * weights['llm']  # Relevance
        beta = 0.4 * weights['llm']   # Consensus
        gamma = weights['fcd']         # FCD score
        if VERBOSE:
            print(f"   📐 Fórmula: {alpha:.2f}×relevance + {beta:.2f}×consensus + {gamma:.2f}×fcd_score")
    else:
        alpha = 0.6
        beta = 0.4
        gamma = 0.0
        if VERBOSE:
            print(f"   📐 Fórmula: {alpha:.2f}×relevance + {beta:.2f}×consensus")

    # Agrupar por título (deduplicar)
    item_groups = {}

    for item in validated_items:
        title = item['title']
        if title not in item_groups:
            item_groups[title] = {
                'mentions': [],
                'max_relevance': 0,
                'max_fcd_score': 0,
                'item_data': item  # Guardar primer item para metadata
            }

        item_groups[title]['mentions'].append(item)
        item_groups[title]['max_relevance'] = max(
            item_groups[title]['max_relevance'],
            item['relevance_score']
        )

        # Si tiene FCD score, guardar el máximo
        if 'fcd_score' in item:
            item_groups[title]['max_fcd_score'] = max(
                item_groups[title]['max_fcd_score'],
                item['fcd_score']
            )

    # Calcular rank_score
    ranked_items = []
    max_mentions = max(len(g['mentions']) for g in item_groups.values())

    for title, data in item_groups.items():
        # Relevancia: máxima de todas las menciones
        relevance = data['max_relevance']

        # Consenso: menciones / max_mentions (solo para LLMs, no FCD)
        llm_mentions = [m for m in data['mentions'] if m['llm_source'] != 'fcd']
        consensus = len(llm_mentions) / max_mentions if max_mentions > 0 else 0

        # FCD score: máximo si existe
        fcd_score = data['max_fcd_score']

        # Determinar fuentes
        sources = [m['llm_source'] for m in data['mentions']]
        has_llm = len(llm_mentions) > 0
        has_fcd = fcd_score > 0

        # Rank score con lógica adaptativa:
        # - Items SOLO de FCD (sin LLM) → NO reciben boost, solo relevance
        # - Items de LLM (con o sin FCD) → fórmula completa
        if not has_llm and has_fcd:
            # Item SOLO de FCD → no aplicar boost, solo relevance (penalizado)
            rank_score = alpha * relevance * 0.7  # Penalización 30% por no tener LLM
        else:
            # Item tiene LLM → fórmula normal (FCD es opcional boost)
            rank_score = alpha * relevance + beta * consensus + (gamma * fcd_score if has_fcd else 0)

        item = data['item_data']
        ranked_items.append({
            'title': title,
            'item_id': item['item_id'],
            'description': item['description'],
            'rank_score': rank_score,
            'relevance': relevance,
            'consensus': consensus,
            'fcd_score': fcd_score if fcd_score > 0 else None,
            'mention_count': len(data['mentions']),
            'llm_sources': sources,
            'llm_reasonings': [m['llm_reasoning'] for m in data['mentions']]
        })

    # Ordenar por rank_score descendente
    ranked_items.sort(key=lambda x: x['rank_score'], reverse=True)

    print(f"   ✅ {len(ranked_items)} items únicos rankeados")
    print(f"      Top-3:")
    for i, item in enumerate(ranked_items[:3], 1):
        fcd_info = f", fcd={item['fcd_score']:.3f}" if item['fcd_score'] else ""
        print(f"         {i}. {item['title'][:50]:50s} (score={item['rank_score']:.3f}, mentions={item['mention_count']}{fcd_info})")

    return ranked_items



### Componente 5: Diversificación (MMR)

Finalmente, para evitar una lista de resultados redundante (ej. "Top 10: Batman Begins, The Dark Knight, The Dark Knight Rises..."), aplicamos **Maximal Marginal Relevance (MMR)**.

MMR construye la lista final iterativamente:

1.  **Iteración 1:** Selecciona el ítem con el `rank_score` más alto.
2.  **Iteraciones 2-k:** Para cada candidato restante, calcula un `mmr_score` que balancea qué tan bueno es (`relevance`) vs. qué tan *diferente* es de los ya seleccionados (`diversity`).
3.  Selecciona el ítem que maximiza esta fórmula.

---

#### Fórmula 4: Maximal Marginal Relevance (MMR)

$$ MMR(d) = \\lambda \\times Rel(d) - (1 - \\lambda) \\times \\max_{s \\in S} sim(d, s) $$

Donde:
-   $d$ es el ítem candidato.
-   $S$ es el conjunto de ítems ya seleccionados.
-   $Rel(d)$ es el `rank_score` del candidato (de la Fórmula 3).
-   $sim(d, s)$ es la similitud de coseno entre los *embeddings* de $d$ y $s$.
-   $\\lambda$ (lambda) es el parámetro de balance. Lo fijamos en **0.7**, dando 70% de importancia a la relevancia y 30% a la diversidad.

In [ ]:
# ═══════════════════════════════════════════════════════
# COMPONENTE 5: MMR DIVERSIFICATION
# ═══════════════════════════════════════════════════════

def mmr_diversification(ranked_items, embeddings_map, k=10, lambda_param=0.7):
    """
    Diversificación usando Maximal Marginal Relevance (MMR).
    Formula: MMR(d) = λ * Rel(d) - (1-λ) * max_sim(d, S)

    OPTIMIZATION: Usa operaciones vectorizadas (NumPy) si USE_VECTORIZED_MMR=True.
    Beneficio: 5-10x más rápido vs loops manuales.

    Args:
        ranked_items (list): Items rankeados por rank_score
        embeddings_map (dict): {title: embedding}
        k (int): Número de items a seleccionar
        lambda_param (float): Balance relevancia/diversidad (0.7 = 70% relevancia)

    Returns:
        list: Top-K items diversificados
    """
    print(f"\n🎨 Diversificación MMR (k={k}, λ={lambda_param}, vectorizado={USE_VECTORIZED_MMR})...")

    if len(ranked_items) <= k:
        print(f"   ℹ️ Solo hay {len(ranked_items)} items, retornando todos")
        return ranked_items

    # OPTIMIZATION: MMR Vectorizado (5-10x más rápido)
    if USE_VECTORIZED_MMR:
        # Precomputar todos los embeddings como matriz
        titles = [item['title'] for item in ranked_items]
        embeddings_list = []

        for title in titles:
            if title in embeddings_map:
                embeddings_list.append(embeddings_map[title])
            else:
                # Item sin embedding → usar vector zero (será filtrado)
                embeddings_list.append(np.zeros(384))

        embeddings_matrix = np.array(embeddings_list)  # (n_items, 384)

        # Normalizar para cosine similarity (más eficiente)
        norms = np.linalg.norm(embeddings_matrix, axis=1, keepdims=True)
        norms = np.where(norms == 0, 1e-10, norms)
        embeddings_norm = embeddings_matrix / norms

        # Relevance scores
        relevances = np.array([item['rank_score'] for item in ranked_items])

        selected_indices = []
        remaining_indices = list(range(len(ranked_items)))

        # Iteración 1: Seleccionar primer item (mayor rank_score)
        selected_indices.append(remaining_indices.pop(0))
        print(f"   Iter 1: {ranked_items[selected_indices[0]]['title'][:40]:40s} (rank={ranked_items[selected_indices[0]]['rank_score']:.3f})")

        # Iteraciones 2-k
        iteration = 2
        while len(selected_indices) < k and remaining_indices:
            # Embeddings de seleccionados y restantes
            selected_embs = embeddings_norm[selected_indices]  # (n_selected, 384)
            remaining_embs = embeddings_norm[remaining_indices]  # (n_remaining, 384)

            # Matriz de similitudes (vectorizada): (n_remaining, n_selected)
            similarities_matrix = remaining_embs @ selected_embs.T

            # Max similarity por candidato
            max_sims = similarities_matrix.max(axis=1)  # (n_remaining,)

            # Diversity scores
            diversities = 1 - max_sims

            # Relevance scores de restantes
            relevances_remaining = relevances[remaining_indices]

            # MMR scores (vectorizado)
            mmr_scores = lambda_param * relevances_remaining + (1 - lambda_param) * diversities

            # Seleccionar mejor
            best_local_idx = mmr_scores.argmax()
            best_global_idx = remaining_indices.pop(best_local_idx)
            selected_indices.append(best_global_idx)

            best_item = ranked_items[best_global_idx]
            best_mmr = mmr_scores[best_local_idx]

            print(f"   Iter {iteration}: {best_item['title'][:40]:40s} (mmr={best_mmr:.3f}, rank={best_item['rank_score']:.3f})")
            iteration += 1

        selected = [ranked_items[idx] for idx in selected_indices]
        print(f"   ✅ {len(selected)} items seleccionados con diversidad (vectorizado)")
        return selected

    # Fallback: Implementación original (loops)
    selected = []
    candidates = ranked_items.copy()

    # Iteración 1: Seleccionar item con mayor rank_score
    if candidates:
        selected.append(candidates.pop(0))
        print(f"   Iter 1: {selected[0]['title'][:40]:40s} (rank={selected[0]['rank_score']:.3f})")

    # Iteraciones 2-k: Balancear relevancia + diversidad
    iteration = 2
    while len(selected) < k and candidates:
        best_mmr = -1
        best_idx = 0

        for i, candidate in enumerate(candidates):
            # Componente de relevancia
            relevance = candidate['rank_score']

            # Componente de diversidad
            cand_title = candidate['title']
            if cand_title not in embeddings_map:
                # Si no hay embedding, usar diversidad = 1.0
                diversity = 1.0
            else:
                cand_emb = embeddings_map[cand_title]

                # Calcular similitud con todos los ya seleccionados
                similarities = []
                for sel in selected:
                    sel_title = sel['title']
                    if sel_title in embeddings_map:
                        sel_emb = embeddings_map[sel_title]
                        sim = np.dot(cand_emb, sel_emb) / (
                            np.linalg.norm(cand_emb) * np.linalg.norm(sel_emb) + 1e-10
                        )
                        sim = np.clip(sim, -1.0, 1.0)
                        similarities.append(sim)

                max_sim = max(similarities) if similarities else 0.0
                diversity = 1 - max_sim

            # Calcular MMR score
            mmr_score = lambda_param * relevance + (1 - lambda_param) * diversity

            if mmr_score > best_mmr:
                best_mmr = mmr_score
                best_idx = i

        # Seleccionar mejor candidato
        selected_item = candidates.pop(best_idx)
        selected.append(selected_item)
        print(f"   Iter {iteration}: {selected_item['title'][:40]:40s} (mmr={best_mmr:.3f}, rank={selected_item['rank_score']:.3f})")
        iteration += 1

    print(f"   ✅ {len(selected)} items seleccionados con diversidad")
    return selected



## 4. Definición del Pipeline Completo

La función `recommend` une todos los componentes en un solo pipeline ejecutable.

In [ ]:
def semantic_validation(llm_results, query, df, dataset_name, fcd_results=None, weights=None):
    if fcd_results is None: fcd_results = {}
    config = DATASETS[dataset_name]
    threshold = config['threshold']
    external_threshold = config.get('external_threshold', threshold * 0.85)
    allow_external = config.get('allow_external', False)
    query_emb = encoder.encode([query])[0]
    validated_items = []

    # Iterate through LLM results
    for llm_name, recommendations in llm_results.items():
        for rec in recommendations:
            title = rec['recommendation']
            # Standardizing match logic and metadata extraction to use 'item_id'
            matching_items = df[df['title'].str.contains(title, case=False, regex=False, na=False)]

            if len(matching_items) > 0:
                item_data = matching_items.iloc[0]
                item_text = f"{item_data['title']}. {item_data['description']}"
                item_emb = encoder.encode([item_text])[0]
                item_similarity = (np.dot(query_emb, item_emb) / (np.linalg.norm(query_emb) * np.linalg.norm(item_emb) + 1e-10) + 1) / 2

                if item_similarity >= threshold:
                    validated_items.append({
                        'llm_source': llm_name,
                        'title': item_data['title'],
                        'item_id': item_data['item_id'],
                        'description': item_data['description'],
                        'relevance_score': item_similarity,
                        'llm_reasoning': rec['description']
                    })
    return validated_items

def recommend(query, dataset_name='serious_games', top_k=5, user_id=None):
    config = DATASETS[dataset_name]
    weights = calculate_weights(user_id=user_id, dataset_name=dataset_name)
    df = load_data_gamification()
    collection = load_or_create_collection(dataset_name)
    candidates = rag_retrieval(query, collection, top_k=30)
    llm_results = query_all_llms(query, candidates, dataset_name)

    fcd_results = {}
    if user_id is not None:
        try:
            # FCD requires the ratings to be loaded and mapped using standardized loaders
            ratings_df, matrix, u_map, i_map = load_user_ratings(dataset_name)
            fcd_results['fcd'] = fcd_recommendations(user_id, dataset_name, ratings_df, matrix, u_map, i_map, top_k=30)
        except Exception as e:
            print(f"   ☑′ FCD component skipped: {e}")

    validated = semantic_validation(llm_results, query, df, dataset_name, fcd_results=fcd_results)

    if not validated:
        print("☑′ No items validated. Returning top-K from semantic search.")
        fallback = candidates.head(top_k).copy()
        fallback['rank_score'] = fallback['similarity']
        return fallback

    ranked = calculate_ranking(validated, weights=weights)
    return pd.DataFrame(ranked).head(top_k)

## 5. Datos Comparativos (Single-LLM vs Multi-LLM vs Multi-LLM + FCD)

Una parte clave de la investigación es demostrar *por qué* esta arquitectura compleja es mejor que un enfoque más simple.

Para esto, definimos métricas de evaluación estándar:
1.  **Precision@10 (P@10):** ¿Qué proporción de las 10 recomendaciones son relevantes (están en el *ground truth*)? (Mide exactitud).
2.  **Recall@10 (R@10):** ¿Qué proporción del *total* de ítems relevantes logramos encontrar? (Mide cobertura).
3.  **nDCG@10:** ¿Están los ítems más relevantes en las primeras posiciones? (Mide la calidad del orden).
4.  **ILD@10 (Intra-List Diversity):** ¿Qué tan diferentes son semánticamente las 10 recomendaciones entre sí? (Mide diversidad).


In [ ]:
# ═══════════════════════════════════════════════════════
# MÉTRICAS DE EVALUACIÓN
# ═══════════════════════════════════════════════════════

def precision_at_k(recommendations, relevant_items, k=10):
    """
    Precision@K: Exactitud del top-K.
    Mide qué proporción de las K recomendaciones son relevantes.

    Fórmula: Precision@K = |{Recomendaciones relevantes}| / K
    Rango: [0, 1]
    Valor ideal: ≈1 (todas las recomendaciones son relevantes)

    Args:
        recommendations (list): Lista de items recomendados (títulos o IDs)
        relevant_items (set/list): Conjunto de items relevantes (ground truth)
        k (int): Número de recomendaciones a evaluar (default: 10)

    Returns:
        float: Precision@K en rango [0, 1]

    Ejemplo:
        >>> recs = ["Movie A", "Movie B", "Movie C", ...]  # 10 recomendaciones
        >>> relevant = {"Movie A", "Movie C", "Movie G"}  # 3 relevantes
        >>> precision_at_k(recs, relevant, k=10)
        0.2  # 2 de 10 recomendaciones son relevantes
    """
    top_k_recs = recommendations[:k]
    relevant_set = set(relevant_items)

    # Contar cuántas recomendaciones son relevantes
    hits = sum(1 for item in top_k_recs if item in relevant_set)

    precision = hits / k if k > 0 else 0.0
    return precision


def recall_at_k(recommendations, relevant_items, k=10):
    """
    Recall@K: Cobertura de items relevantes.
    Mide qué proporción de los items relevantes fueron recuperados.

    Fórmula: Recall@K = |{Recomendaciones relevantes}| / |{Total relevantes}|
    Rango: [0, 1]
    Valor ideal: ≈1 (recupera todos los items relevantes)

    Args:
        recommendations (list): Lista de items recomendados (títulos o IDs)
        relevant_items (set/list): Conjunto de items relevantes (ground truth)
        k (int): Número de recomendaciones a evaluar (default: 10)

    Returns:
        float: Recall@K en rango [0, 1]

    Ejemplo:
        >>> recs = ["Movie A", "Movie B", "Movie C", ...]  # 10 recomendaciones
        >>> relevant = {"Movie A", "Movie C", "Movie G"}  # 3 relevantes en total
        >>> recall_at_k(recs, relevant, k=10)
        0.667  # Encontró 2 de 3 relevantes (Movie A, Movie C)
    """
    top_k_recs = recommendations[:k]
    relevant_set = set(relevant_items)

    # Contar cuántos relevantes fueron recuperados
    hits = sum(1 for item in top_k_recs if item in relevant_set)

    total_relevant = len(relevant_set)
    recall = hits / total_relevant if total_relevant > 0 else 0.0
    return recall


def ndcg_at_k(recommendations, relevant_items, k=10):
    """
    nDCG@K (Normalized Discounted Cumulative Gain): Calidad del orden.
    Mide si los items más relevantes están en las posiciones superiores.

    Fórmula:
        DCG@K = Σ(i=1 to K) rel_i / log2(i + 1)
        IDCG@K = DCG de la lista ideal (relevantes primero)
        nDCG@K = DCG@K / IDCG@K

    Rango: [0, 1]
    Valor ideal: ≈1 (los más relevantes están arriba)

    Args:
        recommendations (list): Lista de items recomendados (títulos o IDs)
        relevant_items (set/list): Conjunto de items relevantes (ground truth)
        k (int): Número de recomendaciones a evaluar (default: 10)

    Returns:
        float: nDCG@K en rango [0, 1]

    Ejemplo:
        >>> recs = ["Movie A", "Movie B", "Movie C", "Movie D", ...]
        >>> relevant = {"Movie A", "Movie C"}
        >>> ndcg_at_k(recs, relevant, k=10)
        0.85  # Movie A en posición 1 (bueno), Movie C en posición 3 (aceptable)
    """
    top_k_recs = recommendations[:k]
    relevant_set = set(relevant_items)

    # Calcular DCG (Discounted Cumulative Gain)
    dcg = 0.0
    for i, item in enumerate(top_k_recs, start=1):
        relevance = 1.0 if item in relevant_set else 0.0
        dcg += relevance / np.log2(i + 1)

    # Calcular IDCG (Ideal DCG)
    # Lista ideal: todos los relevantes primero, luego los no relevantes
    num_relevant_in_top_k = min(len(relevant_set), k)
    idcg = 0.0
    for i in range(1, num_relevant_in_top_k + 1):
        idcg += 1.0 / np.log2(i + 1)

    # nDCG
    ndcg = dcg / idcg if idcg > 0 else 0.0
    return ndcg


def ild_at_k(recommendations, embeddings_map, k=10):
    """
    ILD@K (Intra-List Diversity): Diversidad semántica.
    Mide qué tan diferentes son las recomendaciones entre sí.

    Fórmula:
        ILD@K = (2 / (K * (K-1))) * Σ Σ (1 - cosine_similarity(i, j))
        Promedio de todas las distancias por pares

    Rango: [0, 1]
    Valor ideal: ≈1 (recomendaciones muy variadas)

    Args:
        recommendations (list): Lista de items recomendados (títulos)
        embeddings_map (dict): {título: embedding vector}
        k (int): Número de recomendaciones a evaluar (default: 10)

    Returns:
        float: ILD@K en rango [0, 1]

    Ejemplo:
        >>> recs = ["Sci-Fi Movie 1", "Sci-Fi Movie 2", "Romance Movie", ...]
        >>> embeddings_map = {"Sci-Fi Movie 1": vec1, ...}
        >>> ild_at_k(recs, embeddings_map, k=10)
        0.45  # Baja diversidad (muchas sci-fi similares)
    """
    top_k_recs = recommendations[:k]

    if k < 2:
        return 0.0  # No se puede calcular diversidad con menos de 2 items

    # Filtrar solo los items que tienen embeddings
    valid_recs = [rec for rec in top_k_recs if rec in embeddings_map]

    if len(valid_recs) < 2:
        return 0.0

    # Calcular todas las distancias por pares
    distances = []
    for i in range(len(valid_recs)):
        for j in range(i + 1, len(valid_recs)):
            emb_i = embeddings_map[valid_recs[i]]
            emb_j = embeddings_map[valid_recs[j]]

            # Similitud coseno
            cosine_sim = np.dot(emb_i, emb_j) / (
                np.linalg.norm(emb_i) * np.linalg.norm(emb_j) + 1e-10
            )
            cosine_sim = np.clip(cosine_sim, -1.0, 1.0)

            # Distancia = 1 - similitud
            distance = 1.0 - cosine_sim
            distances.append(distance)

    # ILD = promedio de todas las distancias
    ild = np.mean(distances) if distances else 0.0
    return ild


def evaluate_recommendations(recommendations, relevant_items, embeddings_map, k=10):
    """
    Evalúa las recomendaciones usando las 4 métricas estándar.

    Args:
        recommendations (list): Lista de items recomendados (títulos)
        relevant_items (set/list): Items relevantes (ground truth)
        embeddings_map (dict): {título: embedding vector}
        k (int): Número de recomendaciones a evaluar (default: 10)

    Returns:
        dict: {
            'precision@k': float,
            'recall@k': float,
            'ndcg@k': float,
            'ild@k': float
        }

    Ejemplo de uso:
        >>> results_df = recommend("sci-fi time travel", 'movies', top_k=10)
        >>> recs = results_df['title'].tolist()
        >>> relevant = {"Interstellar", "Back to the Future", "The Matrix"}
        >>> embeddings_map = {title: encoder.encode([title])[0] for title in recs}
        >>> metrics = evaluate_recommendations(recs, relevant, embeddings_map)
        >>> print(f"Precision@10: {metrics['precision@10']:.3f}")
        >>> print(f"Recall@10: {metrics['recall@10']:.3f}")
        >>> print(f"nDCG@10: {metrics['ndcg@10']:.3f}")
        >>> print(f"ILD@10: {metrics['ild@10']:.3f}")
    """
    metrics = {
        'precision@10': precision_at_k(recommendations, relevant_items, k),
        'recall@10': recall_at_k(recommendations, relevant_items, k),
        'ndcg@10': ndcg_at_k(recommendations, relevant_items, k),
        'ild@10': ild_at_k(recommendations, embeddings_map, k),
    }

    return metrics


def print_evaluation_report(metrics, query=None):
    """
    Imprime un reporte formateado de las métricas de evaluación.

    Args:
        metrics (dict): Resultado de evaluate_recommendations()
        query (str): Query original (opcional, para contexto)
    """
    print("\n" + "="*70)
    print("📊 REPORTE DE EVALUACIÓN")
    print("="*70)

    if query:
        print(f"Query: '{query}'")
        print("-"*70)

    print(f"\n{'Métrica':<20} {'Valor':<10} {'Interpretación'}")
    print("-"*70)

    precision = metrics['precision@10']
    print(f"{'Precision@10':<20} {precision:>6.3f}    ", end="")
    if precision >= 0.7:
        print("✅ Excelente (alta exactitud)")
    elif precision >= 0.5:
        print("✓ Bueno (exactitud aceptable)")
    else:
        print("⚠️ Mejorable (baja exactitud)")

    recall = metrics['recall@10']
    print(f"{'Recall@10':<20} {recall:>6.3f}    ", end="")
    if recall >= 0.7:
        print("✅ Excelente (alta cobertura)")
    elif recall >= 0.5:
        print("✓ Bueno (cobertura aceptable)")
    else:
        print("⚠️ Mejorable (baja cobertura)")

    ndcg = metrics['ndcg@10']
    print(f"{'nDCG@10':<20} {ndcg:>6.3f}    ", end="")
    if ndcg >= 0.8:
        print("✅ Excelente (orden óptimo)")
    elif ndcg >= 0.6:
        print("✓ Bueno (orden aceptable)")
    else:
        print("⚠️ Mejorable (orden subóptimo)")

    ild = metrics['ild@10']
    print(f"{'ILD@10':<20} {ild:>6.3f}    ", end="")
    if ild >= 0.6:
        print("✅ Excelente (alta diversidad)")
    elif ild >= 0.4:
        print("✓ Bueno (diversidad aceptable)")
    else:
        print("⚠️ Mejorable (baja diversidad)")

    print("="*70)



In [ ]:
def test_architecture_comparison():
    """
    Compara 3 arquitecturas ajustadas para SERIOUS GAMES:
    1. Single LLM (solo 1 LLM activo)
    2. Multi-LLM (todos los LLMs activos)
    3. Multi-LLM + FCD (todos los LLMs + Filtrado Colaborativo de intervenciones)
    """
    print("\n" + "="*70)
    print("COMPARACIÓN DE ARQUITECTURAS - SERIOUS GAMES")
    print("="*70)

    # Query de test específica para el dominio
    test_query = "juego gamificado para reforzar la atención selectiva y percepción de señales de advertencia en una mina subterranea"

    # Ground truth: Componentes técnicos reales en el dataset de Serious Games que deberían ser recuperados
    ground_truth = {
        "Selective visual cue filtering",
        "Timed decision-making loops (Cognitive Function)",
        "Risk evaluation checkpoints",
        "Perceptual discrimination of oxygen cues",
        "Visual scanning to detect safe markers",
        "See-Listen integration tasks",
        "Safety indicator meter",
        "Extrinsic rewards for ventilation detection"
    }

    test_user_id = 1  # Usuario con historial en intervention files

    print(f"\n🔍 Query: '{test_query}'")
    print(f"📊 Ground truth: {len(ground_truth)} componentes relevantes")
    print(f"👤 Usuario de test: {test_user_id}")

    results_table = []
    configs = [
        ('Single LLM', False),
        ('Multi-LLM', False),
        ('Multi-LLM+FCD', True)
    ]

    original_llms = ACTIVE_LLMS.copy()

    for config_name, use_fcd in configs:
        print("\n" + "-"*50)
        print(f"EJECUTANDO: {config_name}")
        print("-"*50)

        try:
            start_time = time.time()

            if config_name == 'Single LLM':
                ACTIVE_LLMS.clear()
                first_key = list(original_llms.keys())[0]
                ACTIVE_LLMS[first_key] = original_llms[first_key]
            else:
                ACTIVE_LLMS.clear()
                ACTIVE_LLMS.update(original_llms)

            # Ejecutar recomendación
            current_user = test_user_id if use_fcd else None
            results_df = recommend(query=test_query, dataset_name='serious_games', top_k=10, user_id=current_user)

            latency = time.time() - start_time
            recommendations = results_df['title'].tolist() if not results_df.empty else []

            # Calcular métricas
            # Usamos un dummy map para ILD si no hay embeddings listos, o los generamos al vuelo
            embeddings_map = {rec: encoder.encode([rec])[0] for rec in recommendations}
            metrics = evaluate_recommendations(recommendations, ground_truth, embeddings_map, k=10)

            results_table.append({
                'config': config_name,
                'precision': metrics['precision@10'],
                'ndcg': metrics['ndcg@10'],
                'latency': latency,
                'hits': sum(1 for rec in recommendations if rec in ground_truth)
            })

            print(f"   ✅ {config_name} finalizado. Hits: {results_table[-1]['hits']}/10")

        except Exception as e:
            print(f"   ❌ Error en {config_name}: {e}")

    # Restaurar estado
    ACTIVE_LLMS.clear()
    ACTIVE_LLMS.update(original_llms)

    # Mostrar Tabla Final
    print("\n" + "="*70)
    print(f"{'Configuración':<18} {'P@10':<8} {'nDCG@10':<10} {'Hits':<6} {'Latencia':<10}")
    print("-" * 70)
    for r in results_table:
        print(f"{r['config']:<18} {r['precision']:<8.3f} {r['ndcg']:<10.3f} {r['hits']:<6d} {r['latency']:<10.1f}s")
    print("="*70)

In [ ]:
def test_full_pipeline():
    """
    Test end-to-end del pipeline completo con llamadas reales a LLMs.
    """
    print("\n" + "="*70)
    print("PRUEBA END-TO-END DEL PIPELINE COMPLETO - SERIOUS GAMES")
    print("="*70)
    print("\n⚠️  ADVERTENCIA: Esta prueba hace llamadas REALES a los LLMs activos")
    print(f"   LLMs que se llamarán: {list(ACTIVE_LLMS.keys())}")

    # Dar tiempo al usuario para cancelar
    print("\n   Presiona Ctrl+C en los próximos 2 segundos para cancelar...")
    try:
        time.sleep(2)
    except KeyboardInterrupt:
        print("\n\n❌ Test cancelado por el usuario")
        return

    print("\n▶️  Iniciando test...\n")

    # Test Case 1: Query de Juegos Serios (Construcción)
    print("\n" + "="*70)
    print("TEST CASE 1: Serious Games - Construction Safety")
    print("="*70)

    query_sg = "juego gamificado para reforzar la atención selectiva y percepción de señales de advertencia en una mina subterranea"

    try:
        results_sg = recommend(
            query=query_sg,
            dataset_name='serious_games',
            top_k=5
        )

        print("\n✅ Test 1 EXITOSO")
        print(f"   - Query: '{query_sg}'")
        print(f"   - Recomendaciones obtenidas: {len(results_sg)}")
        display(results_sg)

    except Exception as e:
        print(f"\n❌ Test 1 FALLÓ: {str(e)}")
        import traceback
        traceback.print_exc()

    # Resumen final
    print("\n\n" + "="*70)
    print("RESUMEN DE TESTS END-TO-END")
    print("="*70)
    print("✅ Pipeline completo funciona correctamente")
    print("\nComponentes validados:")
    print("   1. ✅ Balancing (equal weights)")
    print("   2. ✅ Multi-Source Generation (semantic search + LLM calls)")
    print("   3. ✅ Semantic Validation (triple check)")
    print("   4. ✅ Ranking (0.6*relevance + 0.4*consensus)")
    print("   5. ✅ MMR Diversification (λ=0.7 for movies)")
    print("\nFlujo verificado:")
    print("   Balanceo → LLMs → Validación Semántica → Rankeo → MMR ✅")
    print("="*70)


# ▶️  Ejecución del Pipeline Completo

In [ ]:
test_full_pipeline()


PRUEBA END-TO-END DEL PIPELINE COMPLETO - SERIOUS GAMES

⚠️  ADVERTENCIA: Esta prueba hace llamadas REALES a los LLMs activos
   LLMs que se llamarán: ['gemini', 'claude_haiku', 'gpt4o_mini']

   Presiona Ctrl+C en los próximos 2 segundos para cancelar...

▶️  Iniciando test...


TEST CASE 1: Serious Games - Construction Safety

--- DataFrame Head ---


,item_id,item_type,title,description,type_id,type_label,global_item_id,full_text
0,1,dynamic,Progressive hazard recognition levels,Dynamic: Progressive hazard recognition levels...,1,Progression dynamic,dynamic_1,Type: dynamic. Title: Progressive hazard recog...
1,2,dynamic,Timed decision-making loops (Cognitive Function),Dynamic: Timed decision-making loops (Cognitiv...,3,Cognitive Function Dynamic,dynamic_2,Type: dynamic. Title: Timed decision-making lo...
2,3,dynamic,Selective visual cue filtering,Dynamic: Selective visual cue filtering. Type:...,3,Cognitive Function Dynamic,dynamic_3,Type: dynamic. Title: Selective visual cue fil...
3,4,dynamic,Badges for accurate detection,Dynamic: Badges for accurate detection. Type: ...,4,Motivational Dynamic,dynamic_4,Type: dynamic. Title: Badges for accurate dete...
4,5,dynamic,Unlocking deeper mine areas after correct iden...,Dynamic: Unlocking deeper mine areas after cor...,1,Progression dynamic,dynamic_5,Type: dynamic. Title: Unlocking deeper mine ar...



--- DataFrame Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 108 entries, 0 to 107
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   item_id         108 non-null    object
 1   item_type       108 non-null    object
 2   title           107 non-null    object
 3   description     107 non-null    object
 4   type_id         30 non-null     object
 5   type_label      30 non-null     object
 6   global_item_id  108 non-null    object
 7   full_text       108 non-null    object
dtypes: object(8)
memory usage: 6.9+ KB

🤖 Consultando 3 LLMs en paralelo...
You are an expert recommendation system for gamified serious games.

Available components:
1. [26] extrinsic rewards for ventilation detection, time-based ranking, and penalties for unsafe entries - Dynamic: extrinsic rewards for ventilation detection, time-based ranking, and penalties for unsafe entries. Type: Motivational Dynamic
2. [41] Questionnaire

,id,title,description,similarity,rank_score
0,26,"extrinsic rewards for ventilation detection, t...",Dynamic: extrinsic rewards for ventilation det...,0.321620,0.321620
1,41,Questionnaires reinforcing inhibition-based de...,Questionnaires reinforcing inhibition-based de...,0.307244,0.307244
2,12,elective visual cue filterings,Dynamic: elective visual cue filterings. Type:...,0.303956,0.303956
3,3,Selective visual cue filtering,Dynamic: Selective visual cue filtering. Type:...,0.277741,0.277741
4,15,Reward points for correct preventive actions,Dynamic: Reward points for correct preventive ...,0.276907,0.276907




RESUMEN DE TESTS END-TO-END
✅ Pipeline completo funciona correctamente para Serious Games


In [ ]:
from pathlib import Path
Path("/content/chroma_db").mkdir(parents=True, exist_ok=True)


In [ ]:
test_architecture_comparison()


COMPARACIÓN DE ARQUITECTURAS - SERIOUS GAMES

🔍 Query: 'juego gamificado para reforzar la atención selectiva y percepción de señales de advertencia en una mina subterranea'
📊 Ground truth: 8 componentes relevantes
👤 Usuario de test: 1

--------------------------------------------------
EJECUTANDO: Single LLM
--------------------------------------------------

--- DataFrame Head ---


,item_id,item_type,title,description,type_id,type_label,global_item_id,full_text
0,1,dynamic,Progressive hazard recognition levels,Dynamic: Progressive hazard recognition levels...,1,Progression dynamic,dynamic_1,Type: dynamic. Title: Progressive hazard recog...
1,2,dynamic,Timed decision-making loops (Cognitive Function),Dynamic: Timed decision-making loops (Cognitiv...,3,Cognitive Function Dynamic,dynamic_2,Type: dynamic. Title: Timed decision-making lo...
2,3,dynamic,Selective visual cue filtering,Dynamic: Selective visual cue filtering. Type:...,3,Cognitive Function Dynamic,dynamic_3,Type: dynamic. Title: Selective visual cue fil...
3,4,dynamic,Badges for accurate detection,Dynamic: Badges for accurate detection. Type: ...,4,Motivational Dynamic,dynamic_4,Type: dynamic. Title: Badges for accurate dete...
4,5,dynamic,Unlocking deeper mine areas after correct iden...,Dynamic: Unlocking deeper mine areas after cor...,1,Progression dynamic,dynamic_5,Type: dynamic. Title: Unlocking deeper mine ar...



--- DataFrame Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 108 entries, 0 to 107
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   item_id         108 non-null    object
 1   item_type       108 non-null    object
 2   title           107 non-null    object
 3   description     107 non-null    object
 4   type_id         30 non-null     object
 5   type_label      30 non-null     object
 6   global_item_id  108 non-null    object
 7   full_text       108 non-null    object
dtypes: object(8)
memory usage: 6.9+ KB

🤖 Consultando 1 LLMs en paralelo...
You are an expert recommendation system for gamified serious games.

Available components:
1. [26] extrinsic rewards for ventilation detection, time-based ranking, and penalties for unsafe entries - Dynamic: extrinsic rewards for ventilation detection, time-based ranking, and penalties for unsafe entries. Type: Motivational Dynamic
2. [41] Questionnaire

,item_id,item_type,title,description,type_id,type_label,global_item_id,full_text
0,1,dynamic,Progressive hazard recognition levels,Dynamic: Progressive hazard recognition levels...,1,Progression dynamic,dynamic_1,Type: dynamic. Title: Progressive hazard recog...
1,2,dynamic,Timed decision-making loops (Cognitive Function),Dynamic: Timed decision-making loops (Cognitiv...,3,Cognitive Function Dynamic,dynamic_2,Type: dynamic. Title: Timed decision-making lo...
2,3,dynamic,Selective visual cue filtering,Dynamic: Selective visual cue filtering. Type:...,3,Cognitive Function Dynamic,dynamic_3,Type: dynamic. Title: Selective visual cue fil...
3,4,dynamic,Badges for accurate detection,Dynamic: Badges for accurate detection. Type: ...,4,Motivational Dynamic,dynamic_4,Type: dynamic. Title: Badges for accurate dete...
4,5,dynamic,Unlocking deeper mine areas after correct iden...,Dynamic: Unlocking deeper mine areas after cor...,1,Progression dynamic,dynamic_5,Type: dynamic. Title: Unlocking deeper mine ar...



--- DataFrame Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 108 entries, 0 to 107
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   item_id         108 non-null    object
 1   item_type       108 non-null    object
 2   title           107 non-null    object
 3   description     107 non-null    object
 4   type_id         30 non-null     object
 5   type_label      30 non-null     object
 6   global_item_id  108 non-null    object
 7   full_text       108 non-null    object
dtypes: object(8)
memory usage: 6.9+ KB

🤖 Consultando 1 LLMs en paralelo...
You are an expert recommendation system for gamified serious games.

Available components:
1. [26] extrinsic rewards for ventilation detection, time-based ranking, and penalties for unsafe entries - Dynamic: extrinsic rewards for ventilation detection, time-based ranking, and penalties for unsafe entries. Type: Motivational Dynamic
2. [41] Questionnaire

,item_id,item_type,title,description,type_id,type_label,global_item_id,full_text
0,1,dynamic,Progressive hazard recognition levels,Dynamic: Progressive hazard recognition levels...,1,Progression dynamic,dynamic_1,Type: dynamic. Title: Progressive hazard recog...
1,2,dynamic,Timed decision-making loops (Cognitive Function),Dynamic: Timed decision-making loops (Cognitiv...,3,Cognitive Function Dynamic,dynamic_2,Type: dynamic. Title: Timed decision-making lo...
2,3,dynamic,Selective visual cue filtering,Dynamic: Selective visual cue filtering. Type:...,3,Cognitive Function Dynamic,dynamic_3,Type: dynamic. Title: Selective visual cue fil...
3,4,dynamic,Badges for accurate detection,Dynamic: Badges for accurate detection. Type: ...,4,Motivational Dynamic,dynamic_4,Type: dynamic. Title: Badges for accurate dete...
4,5,dynamic,Unlocking deeper mine areas after correct iden...,Dynamic: Unlocking deeper mine areas after cor...,1,Progression dynamic,dynamic_5,Type: dynamic. Title: Unlocking deeper mine ar...



--- DataFrame Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 108 entries, 0 to 107
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   item_id         108 non-null    object
 1   item_type       108 non-null    object
 2   title           107 non-null    object
 3   description     107 non-null    object
 4   type_id         30 non-null     object
 5   type_label      30 non-null     object
 6   global_item_id  108 non-null    object
 7   full_text       108 non-null    object
dtypes: object(8)
memory usage: 6.9+ KB

🤖 Consultando 1 LLMs en paralelo...
You are an expert recommendation system for gamified serious games.

Available components:
1. [26] extrinsic rewards for ventilation detection, time-based ranking, and penalties for unsafe entries - Dynamic: extrinsic rewards for ventilation detection, time-based ranking, and penalties for unsafe entries. Type: Motivational Dynamic
2. [41] Questionnaire